# 報酬設計の実験（自作パックマン・HRA 版 DQN）

発表用に、報酬を「1回に1つの変数だけ変える」比較と、PSO による報酬の探索を自動で行う。
条件ごとのグラフ・重ね合わせ・Q値の振れ幅・結果表・報酬表・PSO の設定表を **Google Drive の `DQN_reward_study/`** に保存する。

- ランタイムは **CPU（ハイメモリ）** を選ぶ（8並列。1本の学習でバッファに約1.4GB使う）。
- 上から順に実行する。学習のセルは終わるまで実行中のままになる（タブを閉じても Pro+ のバックグラウンド実行で続く）。
- 止まったら、もう一度上から実行すれば終わった学習を飛ばして続きから。
- 全体で 1.5〜2 日かかる見込み。途中でも図表は Drive に更新される。

In [ ]:
from google.colab import drive
drive.mount('/content/drive', timeout_ms=600000)
import os
assert os.path.isdir('/content/drive/MyDrive')
OUT = '/content/drive/MyDrive/DQN_reward_study'
os.makedirs(OUT, exist_ok=True)
print('保存先:', OUT, ' CPU:', os.cpu_count())

In [ ]:
!pip install -q gymnasium
# 図表の日本語フォント
!apt-get -qq install -y fonts-noto-cjk > /dev/null && rm -rf ~/.cache/matplotlib

## コード
ソースを書き出すだけのセル。変更は手元のリポジトリで行い、`python build_reward_notebook.py` で再生成する。

In [ ]:
%%writefile pacman_env.py
"""
Pac-Man Custom Gymnasium Environment
カーナビ風経路表示付きパックマン強化学習環境
"""
import numpy as np
import gymnasium as gym
from gymnasium import spaces
from collections import deque
import heapq


# マップ定義 (0=通路, 1=壁, 2=エサ, 3=パワーエサ)
DEFAULT_MAP = [
    [1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1],
    [1,2,2,2,2,2,2,2,2,1,2,2,2,2,2,2,2,2,1],
    [1,3,1,1,2,1,1,1,2,1,2,1,1,1,2,1,1,3,1],
    [1,2,1,1,2,1,1,1,2,1,2,1,1,1,2,1,1,2,1],
    [1,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,1],
    [1,2,1,1,2,1,2,1,1,1,1,1,2,1,2,1,1,2,1],
    [1,2,2,2,2,1,2,2,2,1,2,2,2,1,2,2,2,2,1],
    [1,1,1,1,2,1,1,1,0,1,0,1,1,1,2,1,1,1,1],
    [1,1,1,1,2,1,0,0,0,0,0,0,0,1,2,1,1,1,1],
    [1,1,1,1,2,1,0,1,1,0,1,1,0,1,2,1,1,1,1],
    [0,0,0,0,2,0,0,1,0,0,0,1,0,0,2,0,0,0,0],
    [1,1,1,1,2,1,0,1,1,1,1,1,0,1,2,1,1,1,1],
    [1,1,1,1,2,1,0,0,0,0,0,0,0,1,2,1,1,1,1],
    [1,1,1,1,2,1,0,1,1,1,1,1,0,1,2,1,1,1,1],
    [1,2,2,2,2,2,2,2,2,1,2,2,2,2,2,2,2,2,1],
    [1,2,1,1,2,1,1,1,2,1,2,1,1,1,2,1,1,2,1],
    [1,3,2,1,2,2,2,2,2,0,2,2,2,2,2,1,2,3,1],
    [1,1,2,1,2,1,2,1,1,1,1,1,2,1,2,1,2,1,1],
    [1,2,2,2,2,1,2,2,2,1,2,2,2,1,2,2,2,2,1],
    [1,2,1,1,1,1,1,1,2,1,2,1,1,1,1,1,1,2,1],
    [1,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,1],
    [1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1],
]

GHOST_COLORS = [(255,0,0), (255,182,255), (0,255,255), (255,165,0)]

CELL_EMPTY = 0
CELL_WALL = 1
CELL_DOT = 2
CELL_POWER = 3

ACTION_UP = 0
ACTION_DOWN = 1
ACTION_LEFT = 2
ACTION_RIGHT = 3
DIRS = [(-1,0),(1,0),(0,-1),(0,1)]

# 報酬の要素 (HRA: 要素ごとに別々のQ値ヘッドで学習する)
REWARD_DOT   = 0   # エサ・パワーエサ・エサへの接近・クリア
REWARD_GHOST = 1   # ゴースト衝突/捕食・接近ペナルティ
REWARD_MOVE  = 2   # 時間・ループ・方向維持・未訪問セル
N_REWARD_COMPONENTS = 3
REWARD_COMPONENT_NAMES = ["dot", "ghost", "move"]

# 学習用の報酬の値 (名前: (既定値, 要素, 説明))。PacManEnv(rewards={...}) で上書きできる (PSO で調整: pso_reward_tuning.py)
# ペナルティも正の値で持ち、使う側で引く
REWARD_SPEC = {
    "dot":            (2.0,   REWARD_DOT,   "エサを食べた (基本)"),
    "dot_progress":   (3.0,   REWARD_DOT,   "エサの終盤ボーナス (食べた割合 × この値)"),
    "power":          (5.0,   REWARD_DOT,   "パワーエサを食べた"),
    "approach_dot":   (0.3,   REWARD_DOT,   "最寄りのエサに近づいた"),
    "leave_dot":      (0.1,   REWARD_DOT,   "最寄りのエサから離れた (罰)"),
    "clear":          (100.0, REWARD_DOT,   "全部食べてクリア"),
    "eat_ghost":      (5.0,   REWARD_GHOST, "青ゴーストを食べた"),
    "death":          (25.0,  REWARD_GHOST, "ゴーストにやられた (罰)"),
    "ghost_adjacent": (1.5,   REWARD_GHOST, "ゴーストが隣のマス (罰)"),
    "ghost_near":     (0.5,   REWARD_GHOST, "ゴーストが2マス以内 (罰)"),
    "chase_blue":     (0.3,   REWARD_GHOST, "パワー中にゴーストへ近づいた"),
    "time":           (0.02,  REWARD_MOVE,  "1ステップごと (罰)"),
    "loop":           (2.0,   REWARD_MOVE,  "同じマスに留まり続けた (罰、回数に応じて増える)"),
    "oscillation":    (3.0,   REWARD_MOVE,  "2マスを往復した (罰)"),
    "heading":        (0.15,  REWARD_MOVE,  "同じ向きに進み続けた"),
    "new_cell":       (0.2,   REWARD_MOVE,  "初めてのマスに入った"),
}
OLD_REWARDS = {k: v[0] for k, v in REWARD_SPEC.items()}   # 前回 (2026-09 まで) の値

# 迷路の数え上げ: エサ152個 + パワーエサ4個 = N 156。全部食べたときのゲーム得点 = 152×10 + 4×50 = 1,720
# 報酬はトータルで設計する (先生の指摘): エサ1個 = +10 を単位にし、エサの報酬合計 10N に桁をそろえる
N_PELLETS = 156
# 報酬の整形 (近づく・往復・初めてのマスなど) は前回の比率を保ったまま、エサの単位 (2 → 10) に合わせて5倍する
SHAPING_ITEMS = ["approach_dot", "leave_dot", "ghost_adjacent", "ghost_near", "chase_blue",
                 "loop", "oscillation", "heading", "new_cell"]
DESIGNED_REWARDS = {
    **{k: OLD_REWARDS[k] * 5 for k in SHAPING_ITEMS},
    "dot":          10.0,              # ゲームの得点と同じ。エサの報酬合計 = 10 × 152 = 1,520
    "dot_progress": 0.0,               # 終盤ボーナスはクリアボーナスに置き換える
    "power":        50.0,              # ゲームの得点と同じ
    "eat_ghost":    200.0,             # ゲームの得点と同じ
    "clear":        10.0 * N_PELLETS,  # 1,560 = 10N。0.99^214 ≈ 0.12 を掛けても開始時点でエサ十数個分に見える
    "death":        500.0,             # 10N の約1/3。前回の 25 (エサ12個分) より重いが、10N を超えない
    "time":         0.1,               # 214 ステップ (最短クリア) で −21 (エサ2個分)
}
REWARD_PRESETS = {"old": OLD_REWARDS, "designed": DESIGNED_REWARDS}
DEFAULT_REWARDS = DESIGNED_REWARDS
REWARD_SIGN = {k: -1.0 if "(罰" in v[2] else 1.0 for k, v in REWARD_SPEC.items()}


class Ghost:
    """
    本家パックマンの4体ゴースト行動を再現

    Blinky（赤）  : パックマンの現在位置を直接追跡
    Pinky（ピンク）: パックマンの進行方向4マス先を狙う（先回り）
    Inky（水色）  : Blinkyとパックマンを使った挟み撃ち
    Clyde（橙）   : 8マス以上遠い→追跡、近い→散開コーナーへ逃げる
    """
    # Clyde の散開コーナー（左下）
    CLYDE_SCATTER = (20, 1)
    # 散開モード時の各ゴーストのコーナー
    SCATTER_TARGETS = [
        (1,  17),   # Blinky → 右上
        (1,   1),   # Pinky  → 左上
        (20, 17),   # Inky   → 右下
        (20,  1),   # Clyde  → 左下
    ]

    def __init__(self, row, col, color, ghost_id=0):
        self.start    = (row, col)
        self.pos      = [row, col]
        self.color    = color
        self.ghost_id = ghost_id   # 0:Blinky 1:Pinky 2:Inky 3:Clyde
        self.scared       = False
        self.scared_timer = 0
        self.last_pos     = None

    def reset(self):
        self.pos      = list(self.start)
        self.scared       = False
        self.scared_timer = 0
        self.last_pos     = None

    def _valid_moves(self, grid):
        rows, cols = len(grid), len(grid[0])
        r, c = self.pos
        moves = []
        for dr, dc in DIRS:
            nr, nc = r + dr, c + dc
            if 0 <= nr < rows and 0 <= nc < cols and grid[nr][nc] != CELL_WALL:
                moves.append((nr, nc))
        return moves

    def _get_target(self, pacman_pos, pacman_dir, blinky_pos):
        """本家ルールに基づいてターゲット座標を返す"""
        pr, pc = pacman_pos
        # 進行方向オフセット
        dir_dr = {ACTION_UP:-1, ACTION_DOWN:1, ACTION_LEFT:0,  ACTION_RIGHT:0}
        dir_dc = {ACTION_UP:0,  ACTION_DOWN:0, ACTION_LEFT:-1, ACTION_RIGHT:1}
        dr = dir_dr.get(pacman_dir, 0)
        dc = dir_dc.get(pacman_dir, 0)

        if self.ghost_id == 0:   # Blinky: 現在位置を直接追跡
            return (pr, pc)

        elif self.ghost_id == 1: # Pinky: 進行方向4マス先
            return (pr + dr*4, pc + dc*4)

        elif self.ghost_id == 2: # Inky: 挟み撃ち
            # パックマンの2マス先 → Blinkyからその2倍の延長線上
            mid_r = pr + dr*2
            mid_c = pc + dc*2
            br, bc = blinky_pos
            return (mid_r*2 - br, mid_c*2 - bc)

        else:                    # Clyde: 距離8以上→追跡、近い→左下コーナー
            dist = abs(self.pos[0]-pr) + abs(self.pos[1]-pc)
            if dist >= 8:
                return (pr, pc)
            else:
                return self.CLYDE_SCATTER

    def move(self, grid, pacman_pos, pacman_dir, blinky_pos, rng, mode="classic"):
        """
        mode="bfs"     : 全ゴーストがパックマンを直接BFS追跡（シンプル版）
        mode="classic" : 本家Pac-Manの4体個別ターゲット行動
        """
        valid = self._valid_moves(grid)
        if not valid:
            return

        # 逆戻り禁止（分岐がある場合のみ）
        no_reverse = [p for p in valid if list(p) != self.last_pos]
        candidates = no_reverse if no_reverse else valid

        if self.scared:
            # パワーエサ中: パックマンから最も遠いマスへ逃げる
            farthest_dist = -1
            for p in candidates:
                d = abs(p[0]-pacman_pos[0]) + abs(p[1]-pacman_pos[1])
                if d > farthest_dist:
                    farthest_dist = d
            farthest = [p for p in candidates
                        if abs(p[0]-pacman_pos[0])+abs(p[1]-pacman_pos[1]) == farthest_dist]
            chosen = rng.choice(farthest)
        else:
            # ターゲット決定: bfsモードは常にパックマン直接追跡
            if mode == "bfs":
                target = pacman_pos
            else:
                target = self._get_target(pacman_pos, pacman_dir, blinky_pos)

            rows, cols = len(grid), len(grid[0])
            target = (max(0, min(rows-1, target[0])),
                      max(0, min(cols-1, target[1])))

            path = _bfs_next(grid, tuple(self.pos), target)
            if path and len(path) > 1 and path[1] in candidates:
                chosen = path[1]
            else:
                # BFS失敗 → マンハッタン距離でターゲットに最も近い方向
                candidates.sort(key=lambda p: abs(p[0]-target[0])+abs(p[1]-target[1]))
                chosen = candidates[0]

        self.last_pos = self.pos[:]
        self.pos = list(chosen)

        if self.scared_timer > 0:
            self.scared_timer -= 1
            if self.scared_timer == 0:
                self.scared = False


def _bfs_next(grid, start, goal):
    """BFSで start→goal の最短経路を返す"""
    rows, cols = len(grid), len(grid[0])
    goal = tuple(goal)
    queue = deque([(start, [start])])
    visited = {start}
    while queue:
        pos, path = queue.popleft()
        if pos == goal:
            return path
        r, c = pos
        for dr, dc in DIRS:
            nr, nc = r+dr, c+dc
            if 0 <= nr < rows and 0 <= nc < cols and grid[nr][nc] != CELL_WALL and (nr,nc) not in visited:
                visited.add((nr, nc))
                queue.append(((nr, nc), path+[(nr, nc)]))
    return []


def astar(grid, start, goal):
    """A*アルゴリズムで経路探索"""
    rows, cols = len(grid), len(grid[0])
    if grid[goal[0]][goal[1]] == CELL_WALL:
        return []

    def h(p):
        return abs(p[0]-goal[0]) + abs(p[1]-goal[1])

    open_heap = [(h(start), 0, start, [start])]
    visited = set()

    while open_heap:
        f, g, pos, path = heapq.heappop(open_heap)
        if pos in visited:
            continue
        visited.add(pos)
        if pos == goal:
            return path
        r, c = pos
        for dr, dc in DIRS:
            nr, nc = r+dr, c+dc
            if 0 <= nr < rows and 0 <= nc < cols and grid[nr][nc] != CELL_WALL and (nr,nc) not in visited:
                new_g = g + 1
                heapq.heappush(open_heap, (new_g + h((nr,nc)), new_g, (nr,nc), path+[(nr,nc)]))
    return []


def find_nearest_dot(grid, pos):
    """最近傍のエサをBFSで探す"""
    rows, cols = len(grid), len(grid[0])
    queue = deque([(pos, [pos])])
    visited = {pos}
    while queue:
        (r, c), path = queue.popleft()
        if grid[r][c] in (CELL_DOT, CELL_POWER):
            return (r, c), path
        for dr, dc in DIRS:
            nr, nc = r+dr, c+dc
            if 0 <= nr < rows and 0 <= nc < cols and grid[nr][nc] != CELL_WALL and (nr,nc) not in visited:
                visited.add((nr, nc))
                queue.append(((nr, nc), path+[(nr, nc)]))
    return None, []


class PacManEnv(gym.Env):
    metadata = {"render_modes": ["human", "rgb_array"], "render_fps": 10}

    def __init__(self, render_mode=None, ghost_mode="classic", no_ghosts=False, num_ghosts=1,
                 rewards=None, lives=1):
        """
        ghost_mode="classic" : 本家Pac-Manの4体個別行動（Blinky/Pinky/Inky/Clyde）
        ghost_mode="bfs"     : 全ゴーストがパックマンをBFSで直接追跡（シンプル版）
        no_ghosts=True       : ゴーストなし（カリキュラム学習フェーズ1用）
        num_ghosts           : ゴーストの数 (1〜4, Blinky→Pinky→Inky→Clydeの順に追加)
        rewards              : 学習用の報酬の値の上書き {名前: 値} (REWARD_SPEC 参照)。ゲームの得点は変わらない
        lives                : 残機。捕まるたびに1減り、0でエピソード終了。残っていればパックマンとゴーストを
                               開始位置に戻して続ける (エサはそのまま。本家と同じ)。既定 1 = 1回捕まったら終了
        """
        super().__init__()
        unknown = set(rewards or {}) - set(DEFAULT_REWARDS)
        if unknown:
            raise ValueError(f"unknown reward names: {sorted(unknown)}")
        self.rw = {**DEFAULT_REWARDS, **(rewards or {})}
        self.max_lives = max(1, int(lives))
        self.base_map = [row[:] for row in DEFAULT_MAP]
        self.rows = len(self.base_map)
        self.cols = len(self.base_map[0])
        self.render_mode = render_mode
        self.ghost_mode = ghost_mode
        self.no_ghosts = no_ghosts
        self.num_ghosts = max(1, min(4, num_ghosts))
        self.cell_size = 28
        self.screen = None
        self.clock = None
        self.rng = np.random.default_rng()

        # 観測空間: グリッド状態(4チャネル) + パックマン位置 + ゴースト位置
        #   + 拡張特徴量: 最近エサ距離(1) + ゴースト距離(n) + 最近エサ方向(4) + パワー残り(1)
        #   + 行動別特徴量: 4方向 × (通路/エサの近さ/危険ゴーストの近さ/弱体ゴーストの近さ)
        n = self.num_ghosts
        obs_size = self.rows * self.cols * 4 + 2 + n*2 + (1 + n + 4 + 1) + 4*4
        self.observation_space = spaces.Box(0, 1, shape=(obs_size,), dtype=np.float32)
        self.action_space = spaces.Discrete(4)

        # 経路表示用
        self.planned_path = []
        self.target_pos = None

    def _init_game(self):
        self.grid = [row[:] for row in self.base_map]
        self.pacman = [16, 9]  # スタート位置
        self.pacman_dir = ACTION_RIGHT  # 初期向き
        self.mouth_tick = 0             # 口アニメ用カウンタ
        self.total_dots = sum(1 for r in range(self.rows) for c in range(self.cols) if self.grid[r][c] in (CELL_DOT, CELL_POWER))
        self.dots_eaten = 0
        self.score = 0
        self.power_mode = False
        self.power_timer = 0
        # ゴースト初期位置: ゴーストハウス内の通路セルに配置
        self.ghosts = [
            Ghost(10, 9,  GHOST_COLORS[0], ghost_id=0),  # Blinky（赤）   直接追跡
            Ghost(10, 8,  GHOST_COLORS[1], ghost_id=1),  # Pinky（ピンク）先回り
            Ghost(10, 10, GHOST_COLORS[2], ghost_id=2),  # Inky（水色）   挟み撃ち
            Ghost(12, 9,  GHOST_COLORS[3], ghost_id=3),  # Clyde（橙）    距離で切替
        ][:self.num_ghosts]
        self.steps = 0
        self.planned_path = []
        self.target_pos = None
        self.last_qvalues = None      # Q値可視化用
        self.pos_history = []         # ループ検出①: 位置履歴
        self.dir_history = []         # ループ検出: 方向履歴
        self.visited_cells = set()    # 未訪問セルボーナス用
        self.ghost_paths = [[] for _ in range(self.num_ghosts)]  # ゴースト経路表示用
        self.prev_dot_dist = None     # 報酬シェーピング用
        self.reward_items = {}        # このエピソードの報酬の項目別合計 (info["reward_items"])
        self.lives = self.max_lives
        self._respawned = False
        self.cleared = False

    def set_qvalues(self, qvalues):
        """エージェントのQ値を受け取り描画に使う (numpy配列 shape=[4])"""
        self.last_qvalues = np.array(qvalues, dtype=np.float32)

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        if seed is not None:
            self.rng = np.random.default_rng(seed)
        self._init_game()
        self._update_planned_path()
        return self._get_obs(), {}

    def _bfs_distance(self, start, target_cells):
        """BFSで start から target_cells のいずれかまでの最短距離を返す"""
        from collections import deque
        sr, sc = start
        visited = {(sr, sc)}
        queue = deque([(sr, sc, 0)])
        max_dist = self.rows + self.cols
        while queue:
            r, c, d = queue.popleft()
            if (r, c) in target_cells:
                return d
            for dr, dc in [(-1,0),(1,0),(0,-1),(0,1)]:
                nr, nc = r+dr, c+dc
                if 0<=nr<self.rows and 0<=nc<self.cols and (nr,nc) not in visited and self.grid[nr][nc] != CELL_WALL:
                    visited.add((nr,nc))
                    queue.append((nr, nc, d+1))
        return max_dist

    def _get_obs(self):
        # 4チャネル: 壁/エサ/パワーエサ/ゴースト
        ch_wall = np.array([[1 if c == CELL_WALL else 0 for c in row] for row in self.grid], dtype=np.float32).flatten()
        ch_dot  = np.array([[1 if c == CELL_DOT else 0 for c in row] for row in self.grid], dtype=np.float32).flatten()
        ch_pow  = np.array([[1 if c == CELL_POWER else 0 for c in row] for row in self.grid], dtype=np.float32).flatten()
        gh_map  = np.zeros((self.rows, self.cols), dtype=np.float32)
        for g in self.ghosts:
            gh_map[g.pos[0], g.pos[1]] = 1.0
        ch_ghost = gh_map.flatten()

        pacman_norm = np.array([self.pacman[0]/self.rows, self.pacman[1]/self.cols], dtype=np.float32)
        ghost_norm  = np.array([c for g in self.ghosts for c in [g.pos[0]/self.rows, g.pos[1]/self.cols]], dtype=np.float32)

        # 拡張特徴量
        max_dist = float(self.rows + self.cols)
        pr, pc = self.pacman

        # 最近エサまでのBFS距離（正規化）
        dot_cells = {(r,c) for r in range(self.rows) for c in range(self.cols) if self.grid[r][c] in (CELL_DOT, CELL_POWER)}
        nearest_dot_dist = self._bfs_distance((pr, pc), dot_cells) / max_dist if dot_cells else 0.0

        # 各ゴーストまでのBFS距離（正規化）
        ghost_dists = np.array([
            self._bfs_distance((pr, pc), {(g.pos[0], g.pos[1])}) / max_dist
            for g in self.ghosts
        ], dtype=np.float32)

        # 最近エサへの方向（4方向one-hot: 上下左右）
        dot_dir = np.zeros(4, dtype=np.float32)
        if dot_cells:
            nearest, _ = find_nearest_dot(self.grid, (pr, pc))
            if nearest:
                dr = nearest[0] - pr
                dc = nearest[1] - pc
                if abs(dr) >= abs(dc):
                    dot_dir[0 if dr < 0 else 1] = 1.0  # 上/下
                else:
                    dot_dir[2 if dc < 0 else 3] = 1.0  # 左/右

        # パワーモード残り時間（正規化）
        power_remain = np.array([self.power_timer / 30.0], dtype=np.float32)

        extra = np.concatenate([[nearest_dot_dist], ghost_dists, dot_dir, power_remain])
        return np.concatenate([ch_wall, ch_dot, ch_pow, ch_ghost, pacman_norm, ghost_norm, extra,
                               self._action_features(dot_cells)])

    def _distance_field(self, sources):
        """複数の始点からのBFS距離マップ (到達不能は -1)"""
        dist = [[-1] * self.cols for _ in range(self.rows)]
        queue = deque()
        for r, c in sources:
            if dist[r][c] < 0:
                dist[r][c] = 0
                queue.append((r, c))
        while queue:
            r, c = queue.popleft()
            d = dist[r][c] + 1
            for dr, dc in DIRS:
                nr, nc = r+dr, c+dc
                if (0 <= nr < self.rows and 0 <= nc < self.cols
                        and dist[nr][nc] < 0 and self.grid[nr][nc] != CELL_WALL):
                    dist[nr][nc] = d
                    queue.append((nr, nc))
        return dist

    def _action_features(self, dot_cells):
        """
        各行動(上下左右)の移動先から見た特徴量。
        全体マップの平坦化ベクトルだけではMLPが「どちらへ動くべきか」を読み取りにくいため、
        行動ごとに エサ・ゴーストまでの迷路上の距離 を直接与える。
        壁方向はその場に留まるので現在位置の値を使う。
        """
        dangerous = [tuple(g.pos) for g in self.ghosts if not g.scared and not self.no_ghosts]
        scared    = [tuple(g.pos) for g in self.ghosts if g.scared and not self.no_ghosts]
        fields = [
            (self._distance_field(dot_cells) if dot_cells else None, 20.0),
            (self._distance_field(dangerous) if dangerous else None, 10.0),
            (self._distance_field(scared) if scared else None, 15.0),
        ]

        pr, pc = self.pacman
        feats = np.zeros((4, 4), dtype=np.float32)
        for a, (dr, dc) in enumerate(DIRS):
            nr, nc = pr+dr, pc+dc
            is_open = 0 <= nr < self.rows and 0 <= nc < self.cols and self.grid[nr][nc] != CELL_WALL
            if not is_open:
                nr, nc = pr, pc
            feats[a, 0] = 1.0 if is_open else 0.0
            for k, (field, horizon) in enumerate(fields, start=1):
                if field is None or field[nr][nc] < 0:
                    continue
                # 近いほど1に近い (horizon以上離れていれば0)
                feats[a, k] = 1.0 - min(field[nr][nc], horizon) / horizon
        return feats.flatten()

    def _update_planned_path(self):
        """カーナビ風: 最近傍のエサへのA*経路を計算"""
        pos = tuple(self.pacman)
        target, _ = find_nearest_dot(self.grid, pos)
        if target:
            self.target_pos = target
            self.planned_path = astar(self.grid, pos, target)
        else:
            self.planned_path = []
            self.target_pos = None

    def _update_ghost_paths(self):
        """各ゴーストのパックマンへのBFS経路を計算"""
        target = tuple(self.pacman)
        for i, g in enumerate(self.ghosts):
            if g.scared:
                self.ghost_paths[i] = []
            else:
                self.ghost_paths[i] = _bfs_next(self.grid, tuple(g.pos), target)

    def _give(self, rc, name, scale=1.0):
        """報酬 name を (罰なら負で) 該当要素に加え、エピソード内の項目別合計 (info["reward_items"]) に記録する"""
        v = REWARD_SIGN[name] * self.rw[name] * scale
        rc[REWARD_SPEC[name][1]] += v
        self.reward_items[name] = self.reward_items.get(name, 0.0) + v

    def _handle_ghost_collision(self, g, rc, terminated):
        """ゴーストとの衝突処理。報酬は rc[REWARD_GHOST] に加算し、terminated を返す"""
        if self.no_ghosts:
            return terminated
        if g.scared:
            g.reset()
            self.score += 200
            self._give(rc, "eat_ghost")
        else:
            # ループ罰(-2〜-3/step)より死亡が「安く」ならないよう大きめにする
            self._give(rc, "death")
            self.lives -= 1
            if self.lives <= 0:
                terminated = True
            else:
                self._respawn()
        return terminated

    def _respawn(self):
        """捕まったが残機がある: パックマンとゴーストを開始位置に戻す (エサ・得点はそのまま)"""
        self.pacman = [16, 9]
        self.pacman_dir = ACTION_RIGHT
        for g in self.ghosts:
            g.reset()
        self.power_mode = False
        self.power_timer = 0
        self.pos_history = []
        self.dir_history = []
        self.prev_dot_dist = None
        self._respawned = True

    def _nearest_dot_dist(self):
        """パックマンから最近傍エサまでのBFS距離"""
        pos = tuple(self.pacman)
        _, path = find_nearest_dot(self.grid, pos)
        return len(path) - 1 if path else 999

    def _min_ghost_dist(self):
        """パックマンから最近傍ゴーストまでのマンハッタン距離"""
        return min(abs(g.pos[0]-self.pacman[0]) + abs(g.pos[1]-self.pacman[1]) for g in self.ghosts)

    def step(self, action):
        self.steps += 1
        self.mouth_tick += 1
        dr, dc = DIRS[action]
        nr, nc = self.pacman[0]+dr, self.pacman[1]+dc

        # 報酬は要素ごとに集計し、合計を通常の報酬として返す (HRA用に info にも要素別で入れる)
        rc = np.zeros(N_REWARD_COMPONENTS, dtype=np.float32)
        self._give(rc, "time")  # 時間ペナルティ

        # ── パックマン移動 ──
        prev_pacman = self.pacman[:]
        if 0 <= nr < self.rows and 0 <= nc < self.cols and self.grid[nr][nc] != CELL_WALL:
            self.pacman = [nr, nc]
            self.pacman_dir = action
        else:
            pass  # 壁衝突: 報酬変化なし(0)

        r, c = self.pacman

        # ── ループ検出①: 同一セル滞留 ──
        HISTORY_LEN = 8    # 直近8ステップを監視
        LOOP_THRESH = 3    # 同じセルに3回以上いたらループ
        self.pos_history.append((r, c))
        if len(self.pos_history) > HISTORY_LEN:
            self.pos_history.pop(0)
        if len(self.pos_history) == HISTORY_LEN:
            visit_count = self.pos_history.count((r, c))
            if visit_count >= LOOP_THRESH:
                self._give(rc, "loop", 1 + (visit_count - LOOP_THRESH) * 0.75)  # 回数が多いほど重く

        # ── ループ検出②: 往復パターン（A→B→A→B）を直接検出 ──
        if len(self.pos_history) >= 6:
            h = self.pos_history
            # 直近6ステップが 2セル間の往復かチェック
            if (h[-1] == h[-3] == h[-5] and h[-2] == h[-4] == h[-6]
                    and h[-1] != h[-2]):
                self._give(rc, "oscillation")  # 往復ペナルティ

        # ── 方向維持ボーナス（同じ向きに3連続で進んだら小ボーナス）──
        self.dir_history.append(action)
        if len(self.dir_history) > 4:
            self.dir_history.pop(0)
        if (len(self.dir_history) == 4
                and self.dir_history[-1] == self.dir_history[-2] == self.dir_history[-3]
                and self.pacman != prev_pacman):
            self._give(rc, "heading")

        # ── 未訪問セルへの探索ボーナス ──
        if (r, c) not in self.visited_cells:
            self.visited_cells.add((r, c))
            self._give(rc, "new_cell")  # 新しいセルを訪問したら小さなボーナス

        # ── エサ取得 ──
        if self.grid[r][c] == CELL_DOT:
            self.grid[r][c] = CELL_EMPTY
            self.dots_eaten += 1
            self.score += 10
            # 終盤ほどエサの価値を上げて、最後の数個を取り切る動機を与える
            self._give(rc, "dot")
            self._give(rc, "dot_progress", self.dots_eaten / self.total_dots)
        elif self.grid[r][c] == CELL_POWER:
            self.grid[r][c] = CELL_EMPTY
            self.dots_eaten += 1
            self.score += 50
            self._give(rc, "power")
            self.power_mode = True
            self.power_timer = 40
            for g in self.ghosts:
                g.scared = True
                g.scared_timer = 40

        # ── エサへの接近ボーナス (報酬シェーピング、2ステップ毎に計算) ──
        if self.steps % 2 == 0:
            curr_dist = self._nearest_dot_dist()
            if self.prev_dot_dist is not None and curr_dist < self.prev_dot_dist:
                self._give(rc, "approach_dot")   # エサに近づいた
            elif self.prev_dot_dist is not None and curr_dist > self.prev_dot_dist:
                self._give(rc, "leave_dot")   # エサから離れた
            self.prev_dot_dist = curr_dist

        # ── ゴースト接近ペナルティ ──
        min_dist = self._min_ghost_dist() if not self.no_ghosts else 999
        if not self.power_mode:
            if min_dist == 1:
                self._give(rc, "ghost_adjacent")   # 隣接
            elif min_dist == 2:
                self._give(rc, "ghost_near")   # 2マス以内
        else:
            if min_dist <= 2:
                self._give(rc, "chase_blue")   # パワー中は近づく方が良い

        # ── パワーモード更新 ──
        if self.power_timer > 0:
            self.power_timer -= 1
            if self.power_timer == 0:
                self.power_mode = False

        # ── 衝突判定① パックマンがゴーストのマスへ踏み込んだ ──
        terminated = False
        collided = set()
        self._respawned = False
        for i, g in enumerate(self.ghosts):
            if g.pos == self.pacman:
                terminated = self._handle_ghost_collision(g, rc, terminated)
                collided.add(i)
                if self._respawned:
                    break

        # ── ゴースト移動 ──
        prev_ghost_pos = [g.pos[:] for g in self.ghosts]
        blinky_pos = tuple(self.ghosts[0].pos)  # InkyのターゲットにBlinkyの位置が必要
        for g in self.ghosts:
            g.move(self.grid, tuple(self.pacman), self.pacman_dir, blinky_pos, self.rng,
                   mode=self.ghost_mode)

        # ── 衝突判定② ゴーストがパックマンのマスへ来た / すり抜け ──
        for i, g in enumerate(self.ghosts):
            if i in collided or self._respawned:   # 開始位置に戻った直後は判定しない
                continue
            # ゴーストがパックマスマスへ移動
            direct = (g.pos == self.pacman)
            # すり抜け: パックマンとゴーストが互いの前のマスへ移動
            passthrough = (prev_ghost_pos[i] == self.pacman and list(g.pos) == prev_pacman)
            if direct or passthrough:
                terminated = self._handle_ghost_collision(g, rc, terminated)

        # ── クリア判定 ──
        # 他の報酬(±数点)と桁を揃える。20000のような巨大値はQ値の学習を不安定にする
        if self.dots_eaten >= self.total_dots:
            self._give(rc, "clear")
            self.cleared = True
            terminated = True
        reward = float(rc.sum())

        truncated = self.steps >= (600 if self.no_ghosts else 1000)

        # ── 経路更新: 描画時のみ実行（訓練時はスキップして高速化）──
        if self.render_mode == "human" and self.steps % 5 == 0:
            self._update_planned_path()
            self._update_ghost_paths()

        if self.render_mode == "human":
            self._render_frame()

        return self._get_obs(), reward, terminated, truncated, {"score": self.score,
                                                                "reward_components": rc,
                                                                "reward_items": self.reward_items,
                                                                "cleared": self.cleared,
                                                                "lives": self.lives,
                                                                "dots_eaten": self.dots_eaten}

    def render(self):
        if self.render_mode == "human":
            self._render_frame()
        elif self.render_mode == "rgb_array":
            return self._render_frame()

    def _render_frame(self):
        import pygame
        cs = self.cell_size
        W = self.cols * cs
        H = self.rows * cs + 60  # 情報バー分

        if self.screen is None:
            pygame.init()
            pygame.display.set_caption("Pac-Man RL - カーナビ風経路表示")
            if self.render_mode == "human":
                self.screen = pygame.display.set_mode((W, H))
            else:
                self.screen = pygame.Surface((W, H))
            self.clock = pygame.time.Clock()
            self.font = pygame.font.SysFont("Arial", 14, bold=True)
            self.font_big = pygame.font.SysFont("Arial", 18, bold=True)

        self.screen.fill((0, 0, 0))

        # ── グリッド描画 ──
        for r in range(self.rows):
            for c in range(self.cols):
                x, y = c*cs, r*cs
                cell = self.grid[r][c]
                if cell == CELL_WALL:
                    pygame.draw.rect(self.screen, (30, 80, 200), (x, y, cs, cs))
                    pygame.draw.rect(self.screen, (50, 100, 220), (x+1, y+1, cs-2, cs-2), 1)
                elif cell == CELL_DOT:
                    pygame.draw.circle(self.screen, (255, 220, 150), (x+cs//2, y+cs//2), 3)
                elif cell == CELL_POWER:
                    t = pygame.time.get_ticks() // 300 % 2
                    col = (255, 200, 50) if t == 0 else (255, 100, 0)
                    pygame.draw.circle(self.screen, col, (x+cs//2, y+cs//2), 7)

        # ── カーナビ経路表示 ──
        path = self.planned_path
        if len(path) > 1:
            # 経路を半透明の帯で描画
            surf = pygame.Surface((W, self.rows*cs), pygame.SRCALPHA)
            for i in range(len(path)-1):
                r0,c0 = path[i]
                r1,c1 = path[i+1]
                x0,y0 = c0*cs+cs//2, r0*cs+cs//2
                x1,y1 = c1*cs+cs//2, r1*cs+cs//2
                # 進捗に応じて色変化 (青→緑)
                t = i / max(len(path)-1, 1)
                color = (int(0+100*t), int(180-80*t), int(255-100*t), 160)
                pygame.draw.line(surf, color, (x0,y0), (x1,y1), 6)
            self.screen.blit(surf, (0,0))

            # ウェイポイント矢印
            for i in range(0, len(path)-1, 2):
                r0,c0 = path[i]
                r1,c1 = path[i+1]
                cx,cy = c0*cs+cs//2, r0*cs+cs//2
                dr,dc = r1-r0, c1-c0
                arrow_pts = self._arrow_points(cx, cy, dc*cs//2, dr*cs//2, 5)
                pygame.draw.polygon(self.screen, (0, 255, 180), arrow_pts)

        # 目標地点マーカー
        if self.target_pos:
            tr, tc = self.target_pos
            tx, ty = tc*cs+cs//2, tr*cs+cs//2
            pygame.draw.circle(self.screen, (255, 50, 50), (tx, ty), 8, 3)
            pygame.draw.line(self.screen, (255, 50, 50), (tx, ty-12), (tx, ty-4), 2)

        # ── ゴースト予測経路表示 ──
        ghost_surf = pygame.Surface((W, self.rows*cs), pygame.SRCALPHA)
        for i, (g, path) in enumerate(zip(self.ghosts, self.ghost_paths)):
            if len(path) < 2 or g.scared:
                continue
            base_color = g.color
            for j in range(len(path) - 1):
                r0, c0 = path[j]
                r1, c1 = path[j+1]
                x0, y0 = c0*cs+cs//2, r0*cs+cs//2
                x1, y1 = c1*cs+cs//2, r1*cs+cs//2
                alpha = max(30, 140 - j * 15)
                col = (*base_color, alpha)
                pygame.draw.line(ghost_surf, col, (x0, y0), (x1, y1), 3)
            # 経路先端に小三角
            if len(path) >= 2:
                r0, c0 = path[-2]; r1, c1 = path[-1]
                ex, ey = c1*cs+cs//2, r1*cs+cs//2
                arrow = self._arrow_points(ex, ey,
                    (c1-c0)*cs//3, (r1-r0)*cs//3, 4)
                pygame.draw.polygon(ghost_surf, (*base_color, 200), arrow)
        self.screen.blit(ghost_surf, (0, 0))

        # ── ゴースト描画 ──
        for g in self.ghosts:
            gr, gc = g.pos
            gx, gy = gc*cs+cs//2, gr*cs+cs//2
            color = (100, 100, 255) if g.scared else g.color
            pygame.draw.circle(self.screen, color, (gx, gy), cs//2-3)
            pygame.draw.rect(self.screen, color, (gc*cs+3, gr*cs+cs//2, cs-6, cs//2-3))
            # 目
            ew = 4 if not g.scared else 3
            ec = (255,255,255) if not g.scared else (255,255,100)
            pygame.draw.circle(self.screen, ec, (gx-5, gy-3), ew)
            pygame.draw.circle(self.screen, ec, (gx+5, gy-3), ew)
            if not g.scared:
                pygame.draw.circle(self.screen, (0,0,100), (gx-4, gy-3), 2)
                pygame.draw.circle(self.screen, (0,0,100), (gx+6, gy-3), 2)

        # ── パックマン描画 (滑らか・向き対応) ──
        pr, pc = self.pacman
        px, py = pc*cs+cs//2, pr*cs+cs//2
        self._draw_pacman(px, py, cs)

        # ── Q値 方向確率矢印 (カーナビ意思決定表示) ──
        if self.last_qvalues is not None:
            self._draw_qvalue_arrows(px, py, cs)

        # ── 情報バー (カーナビ風) ──
        info_y = self.rows * cs
        pygame.draw.rect(self.screen, (20, 20, 40), (0, info_y, W, 60))
        pygame.draw.line(self.screen, (0, 200, 255), (0, info_y), (W, info_y), 2)

        # スコア
        s = self.font_big.render(f"SCORE: {self.score}", True, (0, 255, 200))
        self.screen.blit(s, (10, info_y+8))

        # 経路情報
        path_len = len(self.planned_path)
        nav_text = f"経路: {path_len}歩  目標: {self.target_pos}" if self.target_pos else "経路探索中..."
        nav_col = (0, 220, 255) if path_len > 0 else (255, 150, 0)
        ns = self.font.render(nav_text, True, nav_col)
        self.screen.blit(ns, (10, info_y+32))

        # パワーモード表示
        if self.power_mode:
            pt = self.font.render(f"** POWER MODE! ({self.power_timer})", True, (255, 220, 0))
            self.screen.blit(pt, (W//2-60, info_y+8))

        # エサ残数
        remain = self.total_dots - self.dots_eaten
        ds = self.font.render(f"残エサ: {remain}/{self.total_dots}", True, (200, 200, 200))
        self.screen.blit(ds, (W-130, info_y+8))

        # ステップ
        ss = self.font.render(f"Step: {self.steps}", True, (150, 150, 150))
        self.screen.blit(ss, (W-130, info_y+30))

        if self.render_mode == "human":
            pygame.event.pump()
            pygame.display.flip()
            self.clock.tick(self.metadata["render_fps"])
        else:
            return np.transpose(np.array(pygame.surfarray.pixels3d(self.screen)), (1, 0, 2))

    def _draw_qvalue_arrows(self, px, py, cs):
        """Q値をsoftmax確率に変換して4方向の矢印で表示"""
        import math, pygame
        qv = self.last_qvalues.astype(np.float64)
        # softmax
        qv -= qv.max()
        exp_q = np.exp(qv)
        probs = exp_q / exp_q.sum()

        # 方向: UP=0,DOWN=1,LEFT=2,RIGHT=3
        # (dy, dx) オフセット, 矢印色
        DIR_INFO = [
            (-1,  0, (255,  80,  80)),   # UP    赤
            ( 1,  0, (255, 200,  50)),   # DOWN  黄
            ( 0, -1, (255, 165,   0)),   # LEFT  オレンジ
            ( 0,  1, ( 80, 220,  80)),   # RIGHT 緑
        ]
        max_len = cs * 1.6   # 最大矢印長

        for i, (dy, dx, color) in enumerate(DIR_INFO):
            p = float(probs[i])
            length = max_len * p
            if length < 3:
                continue

            ex = px + dx * length
            ey = py + dy * length

            # 壁方向なら薄く
            nr = self.pacman[0] + dy
            nc = self.pacman[1] + dx
            is_wall = not (0 <= nr < self.rows and 0 <= nc < self.cols
                           and self.grid[nr][nc] != CELL_WALL)
            alpha = 80 if is_wall else 220

            surf = pygame.Surface((self.cols*cs, self.rows*cs), pygame.SRCALPHA)
            # 矢印の軸
            pygame.draw.line(surf, (*color, alpha),
                             (int(px), int(py)), (int(ex), int(ey)),
                             max(2, int(4 * p + 1)))
            # 矢印の頭
            angle = math.atan2(dy, dx)
            head_size = max(4, int(8 * p + 3))
            tip = (ex, ey)
            left  = (ex - head_size*math.cos(angle-0.5),
                     ey - head_size*math.sin(angle-0.5))
            right = (ex - head_size*math.cos(angle+0.5),
                     ey - head_size*math.sin(angle+0.5))
            pygame.draw.polygon(surf, (*color, alpha), [tip, left, right])
            self.screen.blit(surf, (0, 0))

            # パーセント表示
            pct = int(p * 100)
            if pct >= 5:
                label = self.font.render(f"{pct}%", True, color)
                lx = int(px + dx * (length + 6)) - label.get_width()//2
                ly = int(py + dy * (length + 6)) - label.get_height()//2
                self.screen.blit(label, (lx, ly))

    def _draw_pacman(self, px, py, cs):
        """滑らか・向き対応のパックマンを描画する"""
        import math, pygame
        color_pac = (255, 255, 0) if not self.power_mode else (255, 150, 0)

        # 口の開閉アニメ: 0→25→0度を往復
        mouth_open = abs((self.mouth_tick % 16) - 8) * 3.2  # 0〜25.6度

        # 進行方向 → 口の向き (右=0°, 下=90°, 左=180°, 上=270°)
        dir_angle = {
            ACTION_RIGHT: 0.0,
            ACTION_DOWN:  90.0,
            ACTION_LEFT:  180.0,
            ACTION_UP:    270.0,
        }.get(self.pacman_dir, 0.0)

        # 4倍サイズで描いてスムーズスケールダウン → アンチエイリアス
        SCALE = 4
        radius = (cs // 2 - 2) * SCALE
        surf_size = radius * 2 + SCALE
        pac_surf = pygame.Surface((surf_size, surf_size), pygame.SRCALPHA)
        cx = cy = surf_size // 2

        # ① 黄色の塗り潰し円
        pygame.draw.circle(pac_surf, color_pac, (cx, cy), radius)

        # ② 口の開口部: dir_angle ± mouth_open の扇形を黒で塗り潰す
        if mouth_open > 0.5:
            STEPS = 60
            gap = [(cx, cy)]
            a_start = math.radians(dir_angle - mouth_open)
            a_end   = math.radians(dir_angle + mouth_open)
            for s in range(STEPS + 1):
                a = a_start + (a_end - a_start) * s / STEPS
                gap.append((cx + radius * math.cos(a),
                             cy + radius * math.sin(a)))
            pygame.draw.polygon(pac_surf, (0, 0, 0, 255), gap)

        # ③ スムーズスケールダウン
        disp_r = cs // 2 - 2
        disp_d = disp_r * 2
        small = pygame.transform.smoothscale(pac_surf, (disp_d, disp_d))
        self.screen.blit(small, (px - disp_r, py - disp_r))

    def _arrow_points(self, x, y, dx, dy, size):
        import math
        angle = math.atan2(dy, dx)
        pts = [
            (x + dx, y + dy),
            (x + dx - size*math.cos(angle-0.4), y + dy - size*math.sin(angle-0.4)),
            (x + dx - size*math.cos(angle+0.4), y + dy - size*math.sin(angle+0.4)),
        ]
        return pts

    def close(self):
        if self.screen is not None:
            import pygame
            pygame.quit()
            self.screen = None

In [ ]:
%%writefile dqn_agent.py
"""
DQN Agent for Pac-Man
Double DQN + Experience Replay + Target Network
"""
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from collections import deque
import random


class DQNNetwork(nn.Module):
    def __init__(self, obs_size, n_actions):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_size, 512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, n_actions),
        )

    def forward(self, x):
        return self.net(x)


class HRANetwork(nn.Module):
    """
    Hybrid Reward Architecture (van Seijen et al., 2017) 用ネットワーク。
    共有の特徴抽出部 + 報酬要素ごとのQ値ヘッド。forward は全ヘッドの合計Q値を返すので
    行動選択・Q値表示は通常のDQNと同じ扱いができる。
    """
    def __init__(self, obs_size, n_actions, n_heads):
        super().__init__()
        self.n_actions = n_actions
        self.n_heads = n_heads
        self.trunk = nn.Sequential(
            nn.Linear(obs_size, 512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
        )
        self.heads = nn.ModuleList([
            nn.Sequential(nn.Linear(256, 128), nn.ReLU(), nn.Linear(128, n_actions))
            for _ in range(n_heads)
        ])

    def head_q(self, x):
        """要素別Q値: shape [batch, n_heads, n_actions]"""
        h = self.trunk(x)
        return torch.stack([head(h) for head in self.heads], dim=1)

    def forward(self, x):
        return self.head_q(x).sum(dim=1)


class ReplayBuffer:
    def __init__(self, capacity=100000):
        self.buffer = deque(maxlen=capacity)

    def push(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, done))

    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        states, actions, rewards, next_states, dones = zip(*batch)
        return (
            np.array(states, dtype=np.float32),
            np.array(actions, dtype=np.int64),
            np.array(rewards, dtype=np.float32),
            np.array(next_states, dtype=np.float32),
            np.array(dones, dtype=np.float32),
        )

    def __len__(self):
        return len(self.buffer)


class DQNAgent:
    def __init__(self, obs_size, n_actions, lr=1e-4, gamma=0.99,
                 epsilon_start=1.0, epsilon_end=0.05, epsilon_decay=0.995,
                 batch_size=64, target_update_freq=1000, decay_per_episode=False):
        """
        decay_per_episode=True : epsilon を end_episode() ごとに減衰
                                 (False は従来通り学習ステップごと。
                                  1エピソード数百ステップあるので数エピソードで探索が終わる点に注意)
        """
        self.n_actions = n_actions
        self.decay_per_episode = decay_per_episode
        self.gamma = gamma
        self.epsilon = epsilon_start
        self.epsilon_end = epsilon_end
        self.epsilon_decay = epsilon_decay
        self.batch_size = batch_size
        self.target_update_freq = target_update_freq
        self.steps = 0

        self.device = self._get_device()
        print(f"Using device: {self.device}")

        self.policy_net = self._build_net(obs_size, n_actions).to(self.device)
        self.target_net = self._build_net(obs_size, n_actions).to(self.device)
        self.target_net.load_state_dict(self.policy_net.state_dict())
        self.target_net.eval()

        self.optimizer = optim.Adam(self.policy_net.parameters(), lr=lr)
        self.buffer = ReplayBuffer()
        self.loss_history = []

    def _build_net(self, obs_size, n_actions):
        return DQNNetwork(obs_size, n_actions)

    @staticmethod
    def _get_device():
        """CUDA → DirectML (Intel/AMD GPU) → CPU の優先順で選択"""
        if torch.cuda.is_available():
            return torch.device("cuda")
        try:
            import torch_directml
            dml = torch_directml.device()
            # 動作確認: 小さいテンソルを作れるか試す
            torch.tensor([1.0]).to(dml)
            return dml
        except Exception:
            pass
        return torch.device("cpu")

    def get_qvalues(self, state):
        """現在の状態のQ値をnumpy配列で返す"""
        state_t = torch.FloatTensor(state).unsqueeze(0).to(self.device)
        with torch.no_grad():
            q = self.policy_net(state_t)
        return q.squeeze(0).cpu().numpy()

    def select_action(self, state, training=True):
        if training and random.random() < self.epsilon:
            return random.randrange(self.n_actions)
        q_values = self.get_qvalues(state)
        return int(np.argmax(q_values))

    def push(self, state, action, reward, next_state, done):
        self.buffer.push(state, action, reward, next_state, done)

    def train_step(self):
        if len(self.buffer) < self.batch_size:
            return None

        states, actions, rewards, next_states, dones = self.buffer.sample(self.batch_size)
        states_t     = torch.FloatTensor(states).to(self.device)
        actions_t    = torch.LongTensor(actions).to(self.device)
        rewards_t    = torch.FloatTensor(rewards).to(self.device)
        next_states_t= torch.FloatTensor(next_states).to(self.device)
        dones_t      = torch.FloatTensor(dones).to(self.device)

        loss = self._compute_loss(states_t, actions_t, rewards_t, next_states_t, dones_t)
        self.optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(self.policy_net.parameters(), 1.0)
        self.optimizer.step()

        self.steps += 1
        if self.steps % self.target_update_freq == 0:
            self.target_net.load_state_dict(self.policy_net.state_dict())

        if not self.decay_per_episode:
            self._decay_epsilon()
        return loss.item()

    def _compute_loss(self, states_t, actions_t, rewards_t, next_states_t, dones_t):
        # Double DQN
        q_values = self.policy_net(states_t).gather(1, actions_t.unsqueeze(1)).squeeze(1)
        with torch.no_grad():
            next_actions = self.policy_net(next_states_t).argmax(dim=1)
            next_q = self.target_net(next_states_t).gather(1, next_actions.unsqueeze(1)).squeeze(1)
            target_q = rewards_t + self.gamma * next_q * (1 - dones_t)
        return nn.SmoothL1Loss()(q_values, target_q)

    def _decay_epsilon(self):
        self.epsilon = max(self.epsilon_end, self.epsilon * self.epsilon_decay)

    def end_episode(self):
        """エピソード終了時に呼ぶ (decay_per_episode=True のときのみ epsilon を減衰)"""
        if self.decay_per_episode:
            self._decay_epsilon()

    def save(self, path):
        torch.save({
            "policy": self.policy_net.state_dict(),
            "optimizer": self.optimizer.state_dict(),
            "epsilon": self.epsilon,
            "steps": self.steps,
            "hra_heads": getattr(self, "n_heads", 0),
        }, path)
        print(f"Model saved: {path}")

    def load(self, path):
        # DirectML は map_location に直接渡せないので CPU 経由でロード
        ckpt = torch.load(path, map_location="cpu", weights_only=False)
        self.policy_net.load_state_dict(ckpt["policy"])
        self.policy_net.to(self.device)
        self.target_net.load_state_dict(ckpt["policy"])
        self.target_net.to(self.device)
        self.optimizer.load_state_dict(ckpt["optimizer"])
        self.epsilon = ckpt["epsilon"]
        self.steps = ckpt["steps"]
        print(f"Model loaded: {path}  (epsilon={self.epsilon:.3f})")


class HRADQNAgent(DQNAgent):
    """
    Hybrid Reward Architecture 版 Double DQN。
    報酬を要素(エサ/ゴースト/移動)に分け、要素ごとのヘッドがそれぞれの報酬だけを予測する。
    各ヘッドは単純な関数を学べばよいので、全部入りの報酬1本を学ぶより安定しやすい。

    push() の reward には要素別報酬のベクトル(長さ n_heads)を渡す。
    次状態の行動は合計Q値 (policy_net) の argmax で全ヘッド共通に選び、
    各ヘッドは target_net のその行動の値でブートストラップする (Double DQN)。
    """
    def __init__(self, obs_size, n_actions, n_heads, **kwargs):
        self.n_heads = n_heads   # _build_net で使うため super().__init__ より前に設定
        super().__init__(obs_size, n_actions, **kwargs)

    def _build_net(self, obs_size, n_actions):
        return HRANetwork(obs_size, n_actions, self.n_heads)

    def _compute_loss(self, states_t, actions_t, rewards_t, next_states_t, dones_t):
        # rewards_t: [batch, n_heads]
        idx = actions_t.view(-1, 1, 1).expand(-1, self.n_heads, 1)
        q_heads = self.policy_net.head_q(states_t).gather(2, idx).squeeze(2)
        with torch.no_grad():
            next_actions = self.policy_net(next_states_t).argmax(dim=1)
            next_idx = next_actions.view(-1, 1, 1).expand(-1, self.n_heads, 1)
            next_q = self.target_net.head_q(next_states_t).gather(2, next_idx).squeeze(2)
            target = rewards_t + self.gamma * next_q * (1 - dones_t).unsqueeze(1)
        return nn.SmoothL1Loss()(q_heads, target)


def load_agent(path, obs_size, n_actions, **kwargs):
    """保存ファイルを見て DQN / HRA のどちらかを作り、重みを読み込んで返す"""
    ckpt = torch.load(path, map_location="cpu", weights_only=False)
    n_heads = ckpt.get("hra_heads", 0)
    if n_heads:
        agent = HRADQNAgent(obs_size, n_actions, n_heads, **kwargs)
    else:
        agent = DQNAgent(obs_size, n_actions, **kwargs)
    agent.load(path)
    return agent

In [ ]:
%%writefile pso_optimizer.py
"""
粒子群最適化 (Particle Swarm Optimization: PSO) 実装
================================================================

【PSO理論】

PSOはKennedy & Eberhartが1995年に提案したメタヒューリスティック最適化手法です。
鳥の群れや魚の群泳といった集団知性(Swarm Intelligence)を模倣します。

■ 基本概念
  - 粒子(Particle): 解空間上の1点(=ハイパーパラメータの組み合わせ)
  - 群れ(Swarm)   : 複数の粒子の集合
  - 適応度(Fitness): その粒子が表す解の良さ(=パックマンの平均スコア)
  - pbest         : 各粒子がこれまでに見つけた最良の位置
  - gbest         : 全粒子を通じて見つかった最良の位置

■ 更新式
  各粒子iの速度vと位置xを以下のルールで更新します:

    v[i] = w * v[i]
          + c1 * r1 * (pbest[i] - x[i])   ← 認知項(自分の経験)
          + c2 * r2 * (gbest   - x[i])    ← 社会項(群れの知識)

    x[i] = x[i] + v[i]

  w  : 慣性重み    (inertia weight)     → 過去の速度を維持する度合い
  c1 : 認知係数    (cognitive coeff.)   → 自己の最良位置へ引き寄せる強さ
  c2 : 社会係数    (social coeff.)      → 群れの最良位置へ引き寄せる強さ
  r1, r2: [0,1]の乱数                   → 確率的な探索を実現

■ 慣性重み w の役割
  w が大きい (>0.9): 広い探索 (exploration) を優先
  w が小さい (<0.4): 局所的な精密化 (exploitation) を優先
  → 線形減衰: w = w_max - (w_max - w_min) * t/T で両立

■ 探索空間
  本実装では以下のハイパーパラメータを最適化します:
    - learning_rate : ニューラルネットの更新ステップ幅
    - epsilon_end   : epsilon-greedyの最終探索率
    - epsilon_decay : epsilonの減衰率（1ステップあたり）
    - gamma         : 割引率（将来の報酬をどれだけ重視するか）

■ 対数スケール変換
  学習率などは桁が変わる量のため、log空間で探索します:
    実際の値 = 10^(粒子の位置)
  例: 粒子位置=-3 → 学習率=0.001

================================================================
"""

import numpy as np


class Particle:
    """1つの粒子: ハイパーパラメータ空間上の点"""

    def __init__(self, bounds, rng):
        """
        bounds: [(min1,max1), (min2,max2), ...] 各次元の探索範囲
        """
        self.dim = len(bounds)
        self.bounds = np.array(bounds, dtype=np.float64)

        # 位置をランダム初期化
        self.pos = rng.uniform(self.bounds[:, 0], self.bounds[:, 1])

        # 速度を探索範囲の10%でランダム初期化
        span = self.bounds[:, 1] - self.bounds[:, 0]
        self.vel = rng.uniform(-span * 0.1, span * 0.1)

        self.pbest_pos = self.pos.copy()
        self.pbest_score = -np.inf

    def update_velocity(self, gbest_pos, w, c1, c2, rng):
        """速度更新式: v = w*v + c1*r1*(pbest-x) + c2*r2*(gbest-x)"""
        r1 = rng.random(self.dim)
        r2 = rng.random(self.dim)
        cognitive = c1 * r1 * (self.pbest_pos - self.pos)
        social    = c2 * r2 * (gbest_pos      - self.pos)
        self.vel  = w * self.vel + cognitive + social

        # 速度クリッピング: 探索範囲の20%以内に制限
        span = self.bounds[:, 1] - self.bounds[:, 0]
        self.vel = np.clip(self.vel, -span * 0.2, span * 0.2)

    def update_position(self):
        """位置更新式: x = x + v (境界でクリップ)"""
        self.pos = np.clip(self.pos + self.vel,
                           self.bounds[:, 0], self.bounds[:, 1])

    def update_pbest(self, score):
        """個人最良解の更新"""
        if score > self.pbest_score:
            self.pbest_score = score
            self.pbest_pos   = self.pos.copy()
            return True
        return False


class PSO:
    """
    粒子群最適化エンジン

    パラメータ:
        n_particles : 粒子数 (多いほど広く探索、計算コスト増)
        n_iterations: 世代数 (多いほど収束精度向上)
        w_start     : 初期慣性重み (大きい=広い探索)
        w_end       : 最終慣性重み (小さい=精密化)
        c1          : 認知係数
        c2          : 社会係数
    """

    def __init__(self, n_particles=10, n_iterations=10,
                 w_start=0.9, w_end=0.4, c1=1.5, c2=1.5,
                 seed=42):
        self.n_particles  = n_particles
        self.n_iterations = n_iterations
        self.w_start      = w_start
        self.w_end        = w_end
        self.c1           = c1
        self.c2           = c2
        self.rng          = np.random.default_rng(seed)

        self.gbest_pos    = None
        self.gbest_score  = -np.inf
        self.history      = []   # (iteration, gbest_score, gbest_params)

    def optimize(self, fitness_fn, bounds, param_names=None, verbose=True):
        """
        fitness_fn  : 粒子の位置を受け取りスコアを返す関数
        bounds      : 各次元の探索範囲リスト
        param_names : パラメータ名リスト（表示用）
        戻り値      : (最良パラメータ dict, 履歴 list)
        """
        if param_names is None:
            param_names = [f"x{i}" for i in range(len(bounds))]

        # 粒子群の初期化
        swarm = [Particle(bounds, self.rng) for _ in range(self.n_particles)]

        if verbose:
            print("\n" + "="*60)
            print("  PSO ハイパーパラメータ最適化 開始")
            print(f"  粒子数: {self.n_particles}  世代数: {self.n_iterations}")
            print(f"  慣性重み: {self.w_start} → {self.w_end}  c1={self.c1}  c2={self.c2}")
            print("="*60)
            print(f"\n  探索空間:")
            for name, (lo, hi) in zip(param_names, bounds):
                print(f"    {name:<20}: [{lo:.4f}, {hi:.4f}]")
            print()

        # 世代ループ
        for t in range(self.n_iterations):
            # 線形減衰慣性重み
            w = self.w_start - (self.w_start - self.w_end) * t / max(self.n_iterations - 1, 1)

            if verbose:
                print(f"[世代 {t+1:2d}/{self.n_iterations}]  w={w:.3f}")

            # 各粒子の評価
            for i, p in enumerate(swarm):
                score = fitness_fn(p.pos)

                # 個人最良更新
                updated = p.update_pbest(score)

                # 群れ最良更新
                if score > self.gbest_score:
                    self.gbest_score = score
                    self.gbest_pos   = p.pos.copy()

                if verbose:
                    mark = " *gbest*" if score == self.gbest_score else (
                           " +pbest+" if updated else "")
                    params_str = "  ".join(
                        f"{n}={v:.4f}" for n, v in zip(param_names, p.pos))
                    print(f"  粒子{i+1:2d}: score={score:6.1f}  [{params_str}]{mark}")

            # 速度・位置の更新
            for p in swarm:
                p.update_velocity(self.gbest_pos, w, self.c1, self.c2, self.rng)
                p.update_position()

            self.history.append({
                "iteration"    : t + 1,
                "gbest_score"  : self.gbest_score,
                "gbest_pos"    : self.gbest_pos.copy(),
                "pbest_scores" : [p.pbest_score for p in swarm],  # 全粒子のpbest
                "cur_scores"   : [p.pbest_score for p in swarm],  # 現世代スコア
            })

            if verbose:
                params_str = "  ".join(
                    f"{n}={v:.4f}" for n, v in zip(param_names, self.gbest_pos))
                print(f"  >> 現在の最良スコア: {self.gbest_score:.1f}  [{params_str}]\n")

        # 結果をdict形式で返す
        best_params = {name: val for name, val in zip(param_names, self.gbest_pos)}

        if verbose:
            print("="*60)
            print("  PSO 最適化完了")
            print(f"  最良スコア: {self.gbest_score:.1f}")
            print("  最良パラメータ:")
            for name, val in best_params.items():
                print(f"    {name:<20}: {val:.6f}")
            print("="*60 + "\n")

        return best_params, self.history

In [ ]:
%%writefile reward_study.py
"""
報酬設計の実験 (自作パックマン, HRA 版 DQN)
================================================================
発表用に「1回に1つの変数だけ変える比較」と「PSO による報酬の探索」を自動で行い、
条件ごとのグラフ・重ね合わせグラフ・Q値の振れ幅・結果の表・設計の根拠の表を --out に保存する。
--out を Google Drive にすれば成果物がそのまま Drive に残る。中断しても、終わった学習は飛ばして続きから。

  python reward_study.py --out /content/drive/MyDrive/DQN_reward_study --workers 8
  python reward_study.py --out <dir> --report-only     # 図表だけ作り直す
  python reward_study.py --out <dir> --quick           # 動作確認 (数エピソード)

実験の流れ (前の実験の結果で次の実験の基準値を決める):
  実験0  報酬セット      前回 (old) / トータル設計 (designed)
  実験1  捕獲ペナルティ  100 / 500 / 1,560 (=10N) / 15,600 (=100N, 大きすぎる例)   他は designed で固定
  実験2  クリアボーナス  0 / 500 / 1,560 (=10N) / 5,000                          death は実験1の最良
  実験3  割引率 γ        0.95 / 0.99 / 0.995                                     報酬は実験2までの最良
  実験4  PSO             PSO_SPACE の6変数を探す (適応度 = 学習後の評価スコア)
  実験5  最終比較        前回 / 単変量比較の最良 / PSO の最良 を同じ条件・同じ seed で
================================================================
"""
import argparse
import concurrent.futures as cf
import copy
import csv
import hashlib
import json
import math
import multiprocessing as mp
import os
import random
import time

import numpy as np

from pacman_env import (PacManEnv, REWARD_SPEC, REWARD_SIGN, OLD_REWARDS, DESIGNED_REWARDS,
                        SHAPING_ITEMS, N_PELLETS, N_REWARD_COMPONENTS, REWARD_COMPONENT_NAMES)

# ── 固定する学習設定 (train.py の既定値と同じ) ──
FIXED = {
    "lr": 5e-4,                  # 学習率 α (Adam)
    "gamma": 0.99,               # 割引率 γ (実験3で比較)
    "epsilon_start": 1.0,
    "epsilon_end": 0.0248,
    "explore_ratio": 0.4,        # 全エピソードの40%で ε が下限に届くよう減衰率を決める
    "batch_size": 128,
    "target_update": 1000,       # ターゲットネットワークの更新間隔 (学習ステップ)
    "buffer": 100000,            # 経験再生バッファ
    "max_steps": 1000,           # 1エピソードの上限 (env 側)
    "num_ghosts": 1,
    "ghost_mode": "classic",
}
EVAL_SEED0 = 900000             # 評価に使う盤面の乱数 (全条件で同じ)

# ── PSO の設定 ──
PSO_SPACE = [  # (報酬の名前, 下限, 上限, 説明)
    ("death",     50.0, 3000.0, "捕獲ペナルティ"),
    ("clear",      0.0, 5000.0, "クリアボーナス"),
    ("power",      0.0,  200.0, "パワーエサ"),
    ("eat_ghost",  0.0, 1000.0, "弱体ゴースト"),
    ("time",       0.0,    1.0, "1ステップの罰"),
    ("shaping",    0.0,    2.0, "整形報酬の倍率 (近づく・往復・初めてのマス など9項目)"),
]
PSO_DEFAULT = {"particles": 8, "iterations": 8, "w_start": 0.9, "w_end": 0.4, "c1": 1.5, "c2": 1.5,
               "train_episodes": 400, "eval_episodes": 20, "seed": 101}

NAMES_JA = {k: v[2] for k, v in REWARD_SPEC.items()}
CONDITION_JA = {  # 報酬表の「与える条件」
    "dot": "エサのマスに入った", "dot_progress": "エサを食べたとき (食べた割合 × 値)", "power": "パワーエサのマスに入った",
    "approach_dot": "2ステップごと、最寄りのエサまでの距離が縮んだ", "leave_dot": "2ステップごと、距離が伸びた",
    "clear": "全156個を食べた (エピソード終了)", "eat_ghost": "パワー中にゴーストと同じマスになった",
    "death": "通常のゴーストと同じマス/すれ違い (エピソード終了)", "ghost_adjacent": "パワー中でなく、ゴーストが隣のマス",
    "ghost_near": "パワー中でなく、ゴーストが2マス先", "chase_blue": "パワー中、ゴーストが2マス以内",
    "time": "毎ステップ", "loop": "直近8ステップで同じマスに3回以上 (回数に応じて重く)",
    "oscillation": "直近6ステップが2マスの往復", "heading": "同じ向きに3回続けて進んだ", "new_cell": "そのエピソードで初めてのマスに入った",
}


# ════════════════════════════════════════════════════════════
#  1回の学習
# ════════════════════════════════════════════════════════════
def job_id(job):
    d = {k: job[k] for k in ("label", "rewards", "gamma", "episodes", "seed", "eval_episodes")}
    if job.get("lives", 1) != 1:
        d["lives"] = job["lives"]
    key = json.dumps(d, sort_keys=True, ensure_ascii=False)
    return f"{job['exp']}_{hashlib.sha1(key.encode()).hexdigest()[:10]}_s{job['seed']}"


def reward_unit(rewards):
    """Q値のスケール: エサ1個の報酬が 1 になるように報酬を割ってから学習する"""
    return rewards["dot"] if rewards.get("dot", 0) > 0 else 10.0


def run_job(job):
    import torch
    torch.set_num_threads(1)
    from dqn_agent import HRADQNAgent
    random.seed(job["seed"]); np.random.seed(job["seed"]); torch.manual_seed(job["seed"])
    run_dir = os.path.join(job["runs_dir"], job_id(job))
    os.makedirs(run_dir, exist_ok=True)
    unit = reward_unit(job["rewards"])
    lives = job.get("lives", 1)
    env = PacManEnv(render_mode=None, ghost_mode=FIXED["ghost_mode"], num_ghosts=FIXED["num_ghosts"],
                    rewards=job["rewards"], lives=lives)
    obs_size = env.observation_space.shape[0]
    explore = max(1, int(job["episodes"] * FIXED["explore_ratio"]))
    agent = HRADQNAgent(obs_size, 4, N_REWARD_COMPONENTS, lr=FIXED["lr"], gamma=job["gamma"],
                        epsilon_start=FIXED["epsilon_start"], epsilon_end=FIXED["epsilon_end"],
                        epsilon_decay=FIXED["epsilon_end"] ** (1.0 / explore), batch_size=FIXED["batch_size"],
                        target_update_freq=FIXED["target_update"], decay_per_episode=True)
    # Q値を測る固定の状態 (ランダムに動いたときの256状態。全条件で同じ)
    probe_env = PacManEnv(render_mode=None, num_ghosts=FIXED["num_ghosts"])
    rng = np.random.default_rng(0)
    o, _ = probe_env.reset(seed=0)
    probe = []
    while len(probe) < 256:
        probe.append(o)
        o, _, te, tr, _ = probe_env.step(int(rng.integers(4)))
        if te or tr:
            o, _ = probe_env.reset(seed=len(probe))
    probe_t = torch.FloatTensor(np.array(probe))

    items = list(REWARD_SPEC)
    fields = ["episode", "score", "cleared", "steps", "lives_left", "return", "epsilon", "loss",
              "q_mean", "q_min", "q_max"] + items
    log_path = os.path.join(run_dir, "log.csv")
    t0 = time.time()
    with open(log_path, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=fields)
        w.writeheader()
        q_stat = (0.0, 0.0, 0.0)
        for ep in range(1, job["episodes"] + 1):
            obs, _ = env.reset(seed=job["seed"] * 100000 + ep)
            ret, losses = 0.0, []
            while True:
                a = agent.select_action(obs, training=True)
                nxt, r, te, tr, info = env.step(a)
                agent.push(obs, a, info["reward_components"] / unit, nxt, float(te or tr))
                loss = agent.train_step()
                if loss is not None:
                    losses.append(loss)
                ret += r
                obs = nxt
                if te or tr:
                    break
            agent.end_episode()
            if ep % 10 == 0 or ep == 1:
                with torch.no_grad():
                    q = agent.policy_net(probe_t).max(dim=1).values.numpy() * unit   # 報酬の単位に戻す
                q_stat = (float(q.mean()), float(q.min()), float(q.max()))
            row = {"episode": ep, "score": info["score"], "cleared": int(info["cleared"]), "steps": env.steps,
                   "lives_left": info["lives"], "return": round(ret, 2), "epsilon": round(agent.epsilon, 4),
                   "loss": round(float(np.mean(losses)), 5) if losses else "",
                   "q_mean": round(q_stat[0], 2), "q_min": round(q_stat[1], 2), "q_max": round(q_stat[2], 2)}
            row.update({k: round(info["reward_items"].get(k, 0.0), 2) for k in items})
            w.writerow(row)
            if ep % 50 == 0:
                f.flush()
    train_sec = time.time() - t0
    ev = evaluate(agent, job["rewards"], job["eval_episodes"], lives)
    if job.get("save_model"):
        agent.save(os.path.join(run_dir, "model.pt"))
    result = {"id": job_id(job), "exp": job["exp"], "label": job["label"], "seed": job["seed"],
              "rewards": job["rewards"], "gamma": job["gamma"], "episodes": job["episodes"], "lives": lives,
              "train_minutes": round(train_sec / 60, 1), "eval": ev}
    with open(os.path.join(run_dir, "result.json"), "w", encoding="utf-8") as f:
        json.dump(result, f, ensure_ascii=False, indent=1)
    return result


def evaluate(agent, rewards, n, lives=1):
    """貪欲方策 (ランダムな行動なし) で n 回プレイ。盤面の乱数は全条件で同じ"""
    env = PacManEnv(render_mode=None, ghost_mode=FIXED["ghost_mode"], num_ghosts=FIXED["num_ghosts"], rewards=rewards,
                    lives=lives)
    scores, clears, steps, item_sum, ends = [], [], [], {}, []
    for i in range(n):
        obs, _ = env.reset(seed=EVAL_SEED0 + i)
        while True:
            obs, _, te, tr, info = env.step(agent.select_action(obs, training=False))
            if te or tr:
                break
        scores.append(info["score"]); clears.append(info["cleared"]); steps.append(env.steps)
        ends.append("clear" if info["cleared"] else ("truncated" if tr and not te else "caught"))
        for k, v in info["reward_items"].items():
            item_sum[k] = item_sum.get(k, 0.0) + v / n
    return {"score_mean": float(np.mean(scores)), "score_std": float(np.std(scores)), "scores": scores,
            "clear_rate": float(np.mean(clears)), "steps_mean": float(np.mean(steps)),
            "reward_items_mean": {k: round(v, 2) for k, v in item_sum.items()},
            "endings": {e: ends.count(e) / n for e in ("clear", "caught", "truncated")}}


def run_jobs(jobs, workers):
    """終わっていない学習だけを並列で実行し、全部の結果を返す"""
    results, todo = {}, []
    for j in jobs:
        p = os.path.join(j["runs_dir"], job_id(j), "result.json")
        if os.path.exists(p):
            with open(p, encoding="utf-8") as f:
                results[job_id(j)] = json.load(f)
        else:
            todo.append(j)
    if todo:
        print(f"  学習 {len(todo)} 本を実行 (済み {len(jobs) - len(todo)} 本, 並列 {workers})", flush=True)
        ctx = mp.get_context("fork") if "fork" in mp.get_all_start_methods() else mp.get_context("spawn")
        with cf.ProcessPoolExecutor(max_workers=workers, mp_context=ctx) as ex:
            futs = {ex.submit(run_job, j): j for j in todo}
            for fu in cf.as_completed(futs):
                r = fu.result()
                results[r["id"]] = r
                print(f"  [{time.strftime('%m/%d %H:%M')}] 完了 {r['exp']} {r['label']} seed{r['seed']}: "
                      f"評価 {r['eval']['score_mean']:.0f} 点  クリア率 {r['eval']['clear_rate']:.0%}  "
                      f"({r['train_minutes']} 分)", flush=True)
    return [results[job_id(j)] for j in jobs]


# ════════════════════════════════════════════════════════════
#  実験の定義
# ════════════════════════════════════════════════════════════
def make_jobs(exp, conditions, args, save_model=True, episodes=None, seeds=None, lives=1):
    """conditions: [(ラベル, rewards, gamma, {変えた変数: 値})]。lives = 残機"""
    jobs = []
    for label, rewards, gamma, changed in conditions:
        for s in seeds or args.seeds:
            jobs.append({"exp": exp, "label": label, "rewards": rewards, "gamma": gamma, "changed": changed,
                         "episodes": episodes or args.episodes, "seed": s, "eval_episodes": args.eval_episodes,
                         "runs_dir": os.path.join(args.out, "runs"), "save_model": save_model, "lives": lives})
    return jobs


def summarize(results, conditions):
    """条件ごとに seed をまとめた数値"""
    out = []
    for label, rewards, gamma, changed in conditions:
        rs = [r for r in results if r["label"] == label]
        ev = [r["eval"]["score_mean"] for r in rs]
        out.append({"label": label, "changed": changed, "rewards": rewards, "gamma": gamma, "n": len(rs),
                    "eval_mean": float(np.mean(ev)), "eval_std": float(np.std(ev)),
                    "clear_rate": float(np.mean([r["eval"]["clear_rate"] for r in rs])),
                    "steps_mean": float(np.mean([r["eval"]["steps_mean"] for r in rs])), "ids": [r["id"] for r in rs]})
    return out


def best_of(summary):
    return max(summary, key=lambda s: (s["eval_mean"], s["clear_rate"]))


def with_reward(base, **kw):
    r = dict(base)
    r.update(kw)
    return r


def pso_rewards(base_rewards, pos):
    r = dict(base_rewards)
    for (name, lo, hi, _), v in zip(PSO_SPACE, pos):
        if name == "shaping":
            for k in SHAPING_ITEMS:
                r[k] = DESIGNED_REWARDS[k] * float(v)
        else:
            r[name] = float(v)
    return r


def pso_position(rewards):
    pos = []
    for name, lo, hi, _ in PSO_SPACE:
        if name == "shaping":
            k = SHAPING_ITEMS[0]
            v = rewards[k] / DESIGNED_REWARDS[k] if DESIGNED_REWARDS[k] else 1.0
        else:
            v = rewards[name]
        pos.append(min(max(v, lo), hi))
    return np.array(pos)


def run_pso(args, base_rewards, gamma):
    """PSO で報酬の値を探す。1世代ごとに状態を保存し、中断しても続きから"""
    from pso_optimizer import Particle
    P = dict(PSO_DEFAULT)
    if args.quick:
        P.update(particles=3, iterations=2, train_episodes=args.episodes, eval_episodes=args.eval_episodes)
    state_path = os.path.join(args.out, "pso_state.json")
    bounds = [(lo, hi) for _, lo, hi, _ in PSO_SPACE]
    if os.path.exists(state_path):
        with open(state_path, encoding="utf-8") as f:
            st = json.load(f)
    else:
        rng = np.random.default_rng(P["seed"])
        swarm = [Particle(bounds, rng) for _ in range(P["particles"])]
        swarm[0].pos = pso_position(base_rewards)   # 粒子0は単変量比較の最良から始める
        st = {"settings": P, "gamma": gamma, "base_rewards": base_rewards, "iteration": 0, "history": [],
              "particles": [{"pos": p.pos.tolist(), "vel": p.vel.tolist(), "pbest_pos": p.pos.tolist(),
                             "pbest": None} for p in swarm], "gbest_pos": None, "gbest": None}
    while st["iteration"] < P["iterations"]:
        t = st["iteration"]
        w = P["w_start"] - (P["w_start"] - P["w_end"]) * t / max(P["iterations"] - 1, 1)
        print(f"\n[PSO 世代 {t + 1}/{P['iterations']}]  w={w:.3f}", flush=True)
        conds = [(f"PSO 世代{t + 1} 粒子{i + 1}", pso_rewards(base_rewards, p["pos"]), gamma, {})
                 for i, p in enumerate(st["particles"])]
        jobs = make_jobs("exp4_pso", conds, args, save_model=False, episodes=P["train_episodes"], seeds=[P["seed"]])
        for j in jobs:
            j["eval_episodes"] = P["eval_episodes"]
        res = run_jobs(jobs, args.workers)
        rng = np.random.default_rng(P["seed"] * 1000 + t)   # 再開しても同じ乱数
        for p, r in zip(st["particles"], res):
            fit = r["eval"]["score_mean"]        # 適応度 F = 学習後の貪欲方策の平均スコア
            st["history"].append({"iteration": t + 1, "pos": p["pos"], "fitness": fit,
                                  "clear_rate": r["eval"]["clear_rate"], "id": r["id"]})
            if p["pbest"] is None or fit > p["pbest"]:
                p["pbest"], p["pbest_pos"] = fit, list(p["pos"])
            if st["gbest"] is None or fit > st["gbest"]:
                st["gbest"], st["gbest_pos"] = fit, list(p["pos"])
        for p in st["particles"]:
            part = Particle(bounds, rng)
            part.pos, part.vel = np.array(p["pos"]), np.array(p["vel"])
            part.pbest_pos = np.array(p["pbest_pos"])
            part.update_velocity(np.array(st["gbest_pos"]), w, P["c1"], P["c2"], rng)
            part.update_position()
            p["pos"], p["vel"] = part.pos.tolist(), part.vel.tolist()
        st["iteration"] = t + 1
        st.setdefault("gbest_by_iter", []).append(st["gbest"])
        with open(state_path, "w", encoding="utf-8") as f:
            json.dump(st, f, ensure_ascii=False, indent=1)
        print(f"  最良の適応度 {st['gbest']:.0f}", flush=True)
        report(args)
    return pso_rewards(base_rewards, st["gbest_pos"]), st


def main_pipeline(args):
    os.makedirs(args.out, exist_ok=True)
    chain = []   # 次の実験の基準をどう決めたか
    # 実験0 + 実験1 (互いに独立なので一緒に回す)
    c0 = [("前回 (old)", OLD_REWARDS, 0.99, {"報酬セット": "前回"}),
          ("トータル設計 (designed)", DESIGNED_REWARDS, 0.99, {"報酬セット": "トータル設計"})]
    c1 = [(f"捕獲ペナルティ −{v:,.0f}", with_reward(DESIGNED_REWARDS, death=v), 0.99, {"death": v})
          for v in (100.0, 500.0, 10.0 * N_PELLETS, 100.0 * N_PELLETS)]
    r01 = run_jobs(make_jobs("exp0", c0, args) + make_jobs("exp1", c1, args), args.workers)
    s1 = summarize(r01, c1)
    b1 = best_of(s1)
    chain.append({"from": "実験1", "chosen": {"death": b1["changed"]["death"]}, "reason":
                  f"評価スコアが最も高かった ({b1['eval_mean']:.0f} 点)"})
    report(args)
    base = with_reward(DESIGNED_REWARDS, death=b1["changed"]["death"])
    # 実験2
    c2 = [(f"クリアボーナス +{v:,.0f}", with_reward(base, clear=v), 0.99, {"clear": v})
          for v in (0.0, 500.0, 10.0 * N_PELLETS, 5000.0)]
    s2 = summarize(run_jobs(make_jobs("exp2", c2, args), args.workers), c2)
    b2 = best_of(s2)
    chain.append({"from": "実験2", "chosen": {"clear": b2["changed"]["clear"]}, "reason":
                  f"評価スコアが最も高かった ({b2['eval_mean']:.0f} 点)"})
    report(args)
    base = with_reward(base, clear=b2["changed"]["clear"])
    # 実験3
    c3 = [(f"γ = {g}", base, g, {"gamma": g}) for g in (0.95, 0.99, 0.995)]
    s3 = summarize(run_jobs(make_jobs("exp3", c3, args), args.workers), c3)
    b3 = best_of(s3)
    gamma = b3["changed"]["gamma"]
    chain.append({"from": "実験3", "chosen": {"gamma": gamma}, "reason": f"評価スコアが最も高かった ({b3['eval_mean']:.0f} 点)"})
    with open(os.path.join(args.out, "chain.json"), "w", encoding="utf-8") as f:
        json.dump({"chain": chain, "univariate_best": {"rewards": base, "gamma": gamma}}, f, ensure_ascii=False, indent=1)
    report(args)
    # 実験4 PSO
    pso_best, _ = run_pso(args, base, gamma)
    # 実験5 最終比較
    c5 = [("前回 (old)", OLD_REWARDS, 0.99, {"報酬": "前回"}),
          ("単変量比較の最良", base, gamma, {"報酬": "単変量比較の最良"}),
          ("PSO の最良", pso_best, gamma, {"報酬": "PSO の最良"})]
    run_jobs(make_jobs("exp5", c5, args), args.workers)
    report(args)
    # 実験6・7 残機 (実験0〜5 はどの条件もクリアできず、ほぼ全エピソードが1回の捕獲で終わったため)
    # 実験2 は1回もクリアしないので全条件が同じ結果になり、比較になっていなかった。
    # そこでクリアボーナスは設計どおり 10N に戻し、他は単変量比較の最良 (捕獲・γ) を使う
    base_l = with_reward(base, clear=10.0 * N_PELLETS)
    chain.append({"from": "実験6・7 の基準", "chosen": {"clear": 10.0 * N_PELLETS, "death": base_l["death"], "gamma": gamma},
                  "reason": "実験2 は全条件クリア 0 回で同じ結果だったため、クリアボーナスは設計値 10N に戻した"})
    with open(os.path.join(args.out, "chain.json"), "w", encoding="utf-8") as f:
        json.dump({"chain": chain, "univariate_best": {"rewards": base, "gamma": gamma},
                   "lives_base": {"rewards": base_l, "gamma": gamma}}, f, ensure_ascii=False, indent=1)
    c6 = [(f"残機 {n}", base_l, gamma, {"lives": n}) for n in (1, 3)]
    c7 = [(f"残機3 クリアボーナス +{v:,.0f}", with_reward(base_l, clear=v), gamma, {"clear": v})
          for v in (0.0, 500.0, 10.0 * N_PELLETS, 5000.0)]
    run_jobs(make_jobs("exp6", c6[:1], args, lives=1) + make_jobs("exp6", c6[1:], args, lives=3)
             + make_jobs("exp7", c7, args, lives=3), args.workers)
    report(args)
    print("\nすべての実験が終わりました:", args.out)


# ════════════════════════════════════════════════════════════
#  図表
# ════════════════════════════════════════════════════════════
def _plt():
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    from matplotlib import font_manager as fm
    names = {f.name for f in fm.fontManager.ttflist}
    for c in ("Noto Sans CJK JP", "IPAexGothic", "IPAGothic", "Yu Gothic", "Meiryo", "MS Gothic"):
        if c in names:
            plt.rcParams["font.family"] = c
            break
    plt.rcParams["font.size"] = 12
    return plt


def read_log(run_dir):
    p = os.path.join(run_dir, "log.csv")
    if not os.path.exists(p):
        return None
    with open(p, encoding="utf-8") as f:
        rows = list(csv.DictReader(f))
    if not rows:
        return None
    return {k: np.array([float(r[k]) if r[k] != "" else np.nan for r in rows]) for k in rows[0]}


def mavg(x, n=100):
    c = np.cumsum(np.insert(np.nan_to_num(x), 0, 0.0))
    i = np.arange(1, len(x) + 1)
    lo = np.maximum(0, i - n)
    return (c[i] - c[lo]) / (i - lo)


def cond_line(rewards, gamma, episodes, seeds, lives=1):
    return (f"条件: 残機 {lives} / 捕獲 −{rewards['death']:,.0f} / クリア +{rewards['clear']:,.0f} / エサ +{rewards['dot']:g} / "
            f"γ={gamma} / 学習率 {FIXED['lr']} / ε 1.0→{FIXED['epsilon_end']} / {episodes:,}エピソード / seed {','.join(map(str, seeds))}")


def _text_width(x):
    """表示幅 (全角は2)"""
    return sum(2 if ord(ch) > 0x2E80 else 1 for ch in str(x))


def table_png(plt, path, header, rows, title, col_widths=None):
    widths = [max(_text_width(c) for c in col) + 2 for col in zip(header, *rows)]
    total = sum(widths)
    fig_w = max(8.0, total * 0.105)
    fig_h = 0.42 * (len(rows) + 1) + 0.7
    fig = plt.figure(figsize=(fig_w, fig_h))
    ax = fig.add_axes([0, 0, 1, 1 - 0.55 / fig_h])
    ax.axis("off")
    t = ax.table(cellText=[list(map(str, r)) for r in rows], colLabels=header, loc="upper center",
                 cellLoc="center", colWidths=col_widths or [w / total for w in widths], bbox=[0, 0, 1, 1])
    t.auto_set_font_size(False)
    t.set_fontsize(11)
    for j in range(len(header)):
        t[0, j].set_facecolor("#dfe7f2")
    fig.suptitle(title, fontsize=13, y=1 - 0.12 / fig_h, va="top")
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)


def fmt_num(v):
    v = float(v)
    return f"{v:,.0f}" if abs(v) >= 100 else f"{v:.3g}"


def write_table(base, header, rows, title, plt):
    with open(base + ".csv", "w", newline="", encoding="utf-8-sig") as f:
        csv.writer(f).writerows([header] + rows)
    with open(base + ".md", "w", encoding="utf-8") as f:
        f.write(f"## {title}\n\n| " + " | ".join(header) + " |\n|" + " --- |" * len(header) + "\n")
        for r in rows:
            f.write("| " + " | ".join(map(str, r)) + " |\n")
    table_png(plt, base + ".png", header, rows, title)


EXP_TITLE = {"exp0": "実験0 報酬セット (前回 vs トータル設計)", "exp1": "実験1 捕獲ペナルティ",
             "exp2": "実験2 クリアボーナス", "exp3": "実験3 割引率 γ", "exp6": "実験6 残機 (1 vs 3)",
             "exp7": "実験7 残機3でのクリアボーナス", "exp5": "実験5 最終比較 (前回 / 単変量の最良 / PSO の最良)"}
EXP_AIM = {"exp0": "報酬セット以外の条件をすべて固定し、報酬の設計を変えた効果だけを検証する",
           "exp1": "捕獲ペナルティ以外の条件をすべて固定し、罰の大きさが学習とQ値の振れ幅に与える影響だけを検証する",
           "exp2": "クリアボーナス以外の条件をすべて固定し、全エサを取る報酬の大きさの影響だけを検証する",
           "exp3": "割引率以外の条件をすべて固定し、γ がクリアボーナスの見え方 (γ^T) を通じて与える影響だけを検証する",
           "exp5": "学習の条件と seed をそろえ、報酬の決め方 (前回・単変量比較・PSO) の違いだけを比べる",
           "exp6": "残機以外の条件をすべて固定し、1回の捕獲で終わるかどうかがクリアまでの学習に与える影響だけを検証する",
           "exp7": "残機3で、クリアボーナス以外の条件をすべて固定し、全エサを取る報酬の大きさの影響だけを検証する"}


def report_experiment(plt, out, exp, results):
    rs = [r for r in results if r["exp"] == exp]
    if not rs:
        return None
    labels = list(dict.fromkeys(r["label"] for r in rs))
    d = os.path.join(out, "figures", exp)
    os.makedirs(d, exist_ok=True)
    runs = os.path.join(out, "runs")
    colors = plt.cm.tab10(np.linspace(0, 1, 10))
    means, rows = {}, []
    for ci, lab in enumerate(labels):
        rr = [r for r in rs if r["label"] == lab]
        logs = [read_log(os.path.join(runs, r["id"])) for r in rr]
        logs = [lg for lg in logs if lg is not None]
        if not logs:
            continue
        L = min(len(lg["score"]) for lg in logs)
        sc = np.array([mavg(lg["score"])[:L] for lg in logs])
        means[lab] = (sc.mean(axis=0), np.array([lg["q_max"][:L] - lg["q_min"][:L] for lg in logs]).mean(axis=0), ci)
        # 1条件1枚
        fig, ax = plt.subplots(figsize=(11, 5.2))
        for s, lg in zip(rr, logs):
            ax.plot(lg["episode"][:L], mavg(lg["score"])[:L], color=colors[ci % 10], alpha=0.35, lw=1,
                    label=f"seed {s['seed']}")
        ax.plot(np.arange(1, L + 1), sc.mean(axis=0), color=colors[ci % 10], lw=2.6, label="seed の平均")
        ax.set_xlabel("エピソード")
        ax.set_ylabel("スコア (直近100エピソードの移動平均)")
        ax.set_title(f"{EXP_TITLE.get(exp, exp)}: {lab}\n{cond_line(rr[0]['rewards'], rr[0]['gamma'], rr[0]['episodes'], [r['seed'] for r in rr], rr[0].get('lives', 1))}",
                     fontsize=11)
        ax.axhline(1720, color="gray", ls=":", lw=1)
        ax.text(0.99, 1720, "全エサ 1,720 点 ", color="gray", fontsize=9, ha="right", va="bottom",
                transform=ax.get_yaxis_transform())
        ax.grid(alpha=0.3)
        ax.legend(fontsize=9, loc="upper left")
        fig.tight_layout()
        fig.savefig(os.path.join(d, f"cond{ci + 1}.png"), dpi=150)
        plt.close(fig)
        last = [np.nanmean(lg["score"][-100:]) for lg in logs]
        qr = [lg["q_max"][-1] - lg["q_min"][-1] for lg in logs]
        loss = [np.nanmean(lg["loss"][-100:]) for lg in logs]
        ev = [r["eval"]["score_mean"] for r in rr]
        rows.append([lab, f"{np.mean(ev):,.0f} ± {np.std(ev):,.0f}",
                     f"{np.mean([r['eval']['clear_rate'] for r in rr]):.0%}",
                     f"{np.mean(last):,.0f}", f"{np.mean([r['eval']['steps_mean'] for r in rr]):,.0f}",
                     f"{np.mean(qr):,.0f}", f"{np.mean(loss):.3f}", len(rr)])
    if not means:
        return None
    # 重ね合わせ
    fig, ax = plt.subplots(figsize=(11, 5.2))
    for lab, (m, _, ci) in means.items():
        ax.plot(np.arange(1, len(m) + 1), m, color=colors[ci % 10], lw=2.2, label=lab)
    ax.set_xlabel("エピソード")
    ax.set_ylabel("スコア (直近100エピソードの移動平均, seed の平均)")
    ax.set_title(f"{EXP_TITLE.get(exp, exp)}: 全条件の重ね合わせ\n{EXP_AIM.get(exp, '')}", fontsize=11)
    ax.grid(alpha=0.3)
    ax.legend(fontsize=10)
    fig.tight_layout()
    fig.savefig(os.path.join(d, "overlay.png"), dpi=150)
    plt.close(fig)
    # Q値の振れ幅
    fig, ax = plt.subplots(figsize=(11, 4.8))
    for lab, (_, q, ci) in means.items():
        ax.plot(np.arange(1, len(q) + 1), q, color=colors[ci % 10], lw=2, label=lab)
    ax.set_xlabel("エピソード")
    ax.set_ylabel("Q値の振れ幅 (最大 − 最小, 報酬の単位)")
    ax.set_title(f"{EXP_TITLE.get(exp, exp)}: Q値の振れ幅\n(固定した256状態での max_a Q の最大と最小の差。罰やボーナスを変えると Q のスケールがどう変わるか)",
                 fontsize=11)
    ax.grid(alpha=0.3)
    ax.legend(fontsize=10)
    fig.tight_layout()
    fig.savefig(os.path.join(d, "q_range.png"), dpi=150)
    plt.close(fig)
    header = ["条件", "評価スコア (平均±標準偏差)", "クリア率", "直近100平均 (学習中)", "平均ステップ",
              "Q値の振れ幅", "損失 (直近100)", "試行数"]
    write_table(os.path.join(d, "results"), header, rows,
                f"{EXP_TITLE.get(exp, exp)}: 結果まとめ (評価 = 学習後の貪欲方策で{rs[0]['eval'].get('n', len(rs[0]['eval']['scores']))}回プレイ)", plt)
    return rows


def report_pso(plt, out):
    p = os.path.join(out, "pso_state.json")
    if not os.path.exists(p):
        return
    with open(p, encoding="utf-8") as f:
        st = json.load(f)
    d = os.path.join(out, "figures", "exp4_pso")
    os.makedirs(d, exist_ok=True)
    H = st["history"]
    P = st["settings"]
    # 収束
    fig, ax = plt.subplots(figsize=(11, 5))
    it = [h["iteration"] for h in H]
    ax.scatter(it, [h["fitness"] for h in H], color="tab:gray", alpha=0.6, label="各粒子の適応度")
    g = st.get("gbest_by_iter", [])
    ax.plot(range(1, len(g) + 1), g, "o-", color="tab:red", lw=2.4, label="群れの最良 (gbest)")
    first0 = [h for h in H if h["iteration"] == 1][:1]
    if first0:
        ax.axhline(first0[0]["fitness"], color="tab:blue", ls="--", label="単変量比較の最良 (世代1 粒子1)")
    ax.set_xlabel("世代")
    ax.set_ylabel("適応度 F = 学習後の平均スコア")
    ax.set_title(f"実験4 PSO の収束\n条件: 粒子 {P['particles']} / 世代 {P['iterations']} / w {P['w_start']}→{P['w_end']} / "
                 f"c1={P['c1']} c2={P['c2']} / 1粒子 {P['train_episodes']}エピソード学習 + {P['eval_episodes']}回評価 / γ={st['gamma']}",
                 fontsize=11)
    ax.grid(alpha=0.3)
    ax.legend()
    fig.tight_layout()
    fig.savefig(os.path.join(d, "convergence.png"), dpi=150)
    plt.close(fig)
    # 変数ごとの粒子の位置
    fig, axes = plt.subplots(2, 3, figsize=(14, 7))
    for k, ((name, lo, hi, ja), ax) in enumerate(zip(PSO_SPACE, axes.flat)):
        ax.scatter(it, [h["pos"][k] for h in H], c=[h["fitness"] for h in H], cmap="viridis", s=30)
        if st.get("gbest_pos"):
            ax.axhline(st["gbest_pos"][k], color="tab:red", lw=1.5)
        ax.set_ylim(lo, hi)
        ax.set_title(f"{ja} ({name})  範囲 {lo:g}〜{hi:g}", fontsize=10)
        ax.set_xlabel("世代")
        ax.grid(alpha=0.3)
    fig.suptitle("実験4 PSO: 各変数の粒子の位置 (色 = 適応度, 赤線 = 最良)", fontsize=13)
    fig.tight_layout()
    fig.savefig(os.path.join(d, "positions.png"), dpi=150)
    plt.close(fig)
    # 設定の表
    rows = [[f"{ja} ({n})", f"{lo:g} 〜 {hi:g}", f"{st['base_rewards'].get(n, 1.0) if n != 'shaping' else 1.0:g}",
             (f"{st['gbest_pos'][k]:.3g}" if st.get("gbest_pos") else "-")] for k, (n, lo, hi, ja) in enumerate(PSO_SPACE)]
    write_table(os.path.join(d, "variables"), ["探索する変数", "範囲", "出発点 (単変量の最良)", "PSO の最良"], rows,
                "実験4 PSO: 探索する変数と結果", plt)
    rows = [["粒子数", P["particles"], "群れの大きさ。並列数 (8) に合わせて1世代を1回の並列実行で評価"],
            ["世代数 (反復回数)", P["iterations"], "粒子数 × 世代数 = 学習回数。計算時間の上限から決めた"],
            ["慣性重み w", f"{P['w_start']} → {P['w_end']} (線形に減らす)", "最初は広く探し、後半は最良の周りを細かく探す"],
            ["c1 (自分の最良へ)", P["c1"], "Kennedy & Eberhart の標準的な値"], ["c2 (群れの最良へ)", P["c2"], "同上。c1 と同じにして偏らせない"],
            ["適応度 F", "学習後の貪欲方策の平均スコア", f"{P['train_episodes']}エピソード学習 → 盤面を固定した{P['eval_episodes']}回のプレイ"],
            ["1粒子の学習 seed", P["seed"], "全粒子で同じにして、報酬の違いだけを比べる"],
            ["速度の上限", "範囲の 20%", "一度に範囲の端まで飛ばないように"]]
    write_table(os.path.join(d, "settings"), ["PSO の設定", "値", "理由"], [list(map(str, r)) for r in rows],
                "実験4 PSO の設定", plt)


def report_rewards(plt, out, results):
    """報酬表 (出来事 / 値 / 与える条件) と、前回・単変量の最良・PSO の最良の比較、得点の内訳"""
    d = os.path.join(out, "figures", "rewards")
    os.makedirs(d, exist_ok=True)
    sets = [("前回", OLD_REWARDS), ("トータル設計", DESIGNED_REWARDS)]
    cp = os.path.join(out, "chain.json")
    if os.path.exists(cp):
        with open(cp, encoding="utf-8") as f:
            sets.append(("単変量比較の最良", json.load(f)["univariate_best"]["rewards"]))
    sp = os.path.join(out, "pso_state.json")
    if os.path.exists(sp):
        with open(sp, encoding="utf-8") as f:
            st = json.load(f)
        if st.get("gbest_pos"):
            sets.append(("PSO の最良", pso_rewards(st["base_rewards"], st["gbest_pos"])))
    rows = []
    for k in REWARD_SPEC:
        sign = "−" if REWARD_SIGN[k] < 0 else "+"
        rows.append([NAMES_JA[k].replace(" (罰)", ""), CONDITION_JA[k]] +
                    [f"{sign}{fmt_num(v[k])}" if v[k] else "0" for _, v in sets])
    write_table(os.path.join(d, "reward_table"), ["出来事", "与える条件"] + [n for n, _ in sets], rows,
                "報酬表 (罰は − で表示。PSO が選んだ値も同じ行に並べる)", plt)
    # 報酬の内訳 (実験5 の評価プレイで、1エピソードあたりどの報酬をいくら得たか)
    rs = [r for r in results if r["exp"] == "exp5"]
    if rs:
        labels = list(dict.fromkeys(r["label"] for r in rs))
        fig, ax = plt.subplots(figsize=(12, 5.5))
        x = np.arange(len(labels))
        pos_b, neg_b = np.zeros(len(labels)), np.zeros(len(labels))
        cmap = plt.cm.tab20(np.linspace(0, 1, 20))
        for i, k in enumerate(REWARD_SPEC):
            v = np.array([np.mean([r["eval"]["reward_items_mean"].get(k, 0.0) for r in rs if r["label"] == lab])
                          for lab in labels])
            if not np.any(v):
                continue
            b = np.where(v >= 0, pos_b, neg_b)
            ax.bar(x, v, bottom=b, color=cmap[i % 20], label=NAMES_JA[k])
            pos_b += np.where(v >= 0, v, 0)
            neg_b += np.where(v < 0, v, 0)
        ax.axhline(0, color="black", lw=0.8)
        ax.set_xticks(x, labels)
        ax.set_ylabel("1エピソードあたりの報酬")
        ax.set_title("実験5: 評価プレイで得た報酬の内訳 (各報酬セットで学習したモデル)", fontsize=12)
        ax.legend(fontsize=8, ncol=2, bbox_to_anchor=(1.01, 1), loc="upper left")
        fig.tight_layout()
        fig.savefig(os.path.join(d, "reward_breakdown.png"), dpi=150)
        plt.close(fig)


def report_design(plt, out):
    """スライドの「状態・行動・報酬の設計」「学習設定」に使う数え上げと計算"""
    d = os.path.join(out, "figures", "design")
    os.makedirs(d, exist_ok=True)
    env = PacManEnv(render_mode=None, num_ghosts=FIXED["num_ghosts"])
    env.reset(seed=0)
    R, C = env.rows, env.cols
    write_table(os.path.join(d, "state"), ["状態の要素", "何を表すか", "数"], [
        ["盤面 4チャネル", "壁 / エサ / パワーエサ / ゴースト の有無 (22×19 マスごと)", f"{R}×{C}×4 = {R * C * 4:,}"],
        ["パックマンの位置", "行・列 (0〜1 に正規化)", "2"], ["ゴーストの位置", "行・列 × ゴースト1体", "2"],
        ["最寄りのエサまでの距離", "BFS 距離", "1"], ["ゴーストまでの距離", "1体分", "1"],
        ["最寄りのエサの方向", "上下左右", "4"], ["パワーの残り時間", "0〜1", "1"],
        ["行動別の特徴", "4方向 × (通路か / エサの近さ / 危険なゴーストの近さ / 弱いゴーストの近さ)", "16"],
        ["合計 (ネットワークの入力)", "", f"{env.observation_space.shape[0]:,}"]],
        "状態の構成 (DQN の入力ベクトル)", plt)
    T_min = 214
    rows = []
    for g in (0.95, 0.99, 0.995):
        rows.append([g] + [f"{g ** T:.3f}" for T in (100, T_min, 300, 500, 1000)] +
                    [f"{10 * N_PELLETS * g ** T_min:,.0f}"])
    write_table(os.path.join(d, "gamma_T"), ["γ", "γ^100", f"γ^{T_min} (最短クリア)", "γ^300", "γ^500", "γ^1000 (上限)",
                                             f"クリア +{10 * N_PELLETS:,} の開始時点の価値"], rows,
                "割引率とクリアまでのステップ数 T: γ^T (クリアボーナスが割引で見えなくならないか)", plt)
    rows = [["lr", "学習率 α (Adam)", FIXED["lr"]], ["gamma", "割引率 γ", "0.99 (実験3で比較)"],
            ["epsilon_start → end", "探索率 ε (最初 → 最後)", f"{FIXED['epsilon_start']} → {FIXED['epsilon_end']}"],
            ["epsilon_decay", "ε の減衰", f"エピソードごとに掛ける。全体の{int(FIXED['explore_ratio'] * 100)}%で下限に届く値"],
            ["batch_size", "1回の学習で使う経験の数", FIXED["batch_size"]], ["buffer", "経験再生バッファ", f"{FIXED['buffer']:,}"],
            ["target_update", "ターゲットネットワークの更新間隔", f"{FIXED['target_update']:,} 学習ステップ"],
            ["network", "ネットワーク", f"入力 {env.observation_space.shape[0]:,} → 512 → 256 (共有) → 頭ごとに 128 → 4 (ReLU)"],
            ["heads", "HRA の頭", f"{N_REWARD_COMPONENTS}つ ({' / '.join(REWARD_COMPONENT_NAMES)})。行動は3つの頭の Q の合計で選ぶ"],
            ["q_scale", "Q値のスケール", "報酬をエサ1個の報酬で割って学習 (エサ1個 = 1)。Q の初期値はネットワークの初期値 (≈0)"],
            ["loss", "損失 / 更新式", "Double DQN, Huber 損失, 勾配を 1.0 で切る"],
            ["max_steps", "1エピソードの上限", f"{FIXED['max_steps']:,} ステップ (または全エサ・捕獲で終了)"],
            ["actions", "行動", "4 (上下左右)。壁に向かうとその場に留まる"]]
    write_table(os.path.join(d, "hyperparameters"), ["変数名", "日本語名", "値"], [list(map(str, r)) for r in rows],
                "ハイパーパラメータ一覧 (全実験で固定。γ は実験3でのみ変える)", plt)
    facts = {"rows": R, "cols": C, "pellets": 152, "power_pellets": 4, "N": N_PELLETS,
             "score_all_pellets": 152 * 10 + 4 * 50, "score_all_pellets_plus_ghost_each_power": 152 * 10 + 4 * 50 + 4 * 200,
             "steps_shortest_clear_no_ghost": T_min, "reward_total_dots_designed": 10 * 152,
             "note": "最短クリアは BFS で最寄りのエサを順に取る動き (ゴーストなし) の実測値"}
    with open(os.path.join(d, "facts.json"), "w", encoding="utf-8") as f:
        json.dump(facts, f, ensure_ascii=False, indent=1)


def report_endings(plt, out, results):
    """エピソードの終わり方 (クリア / 捕獲 / 1,000ステップで打ち切り) を条件ごとに集計する"""
    groups = {}
    for r in results:
        if r["exp"] == "exp4_pso":
            continue
        lg = read_log(os.path.join(out, "runs", r["id"]))
        if lg is None:
            continue
        st, cl = lg["steps"][-100:], lg["cleared"][-100:]
        tr_trunc = (cl == 0) & (st >= FIXED["max_steps"])
        tr = (cl.mean(), ((cl == 0) & ~tr_trunc).mean(), tr_trunc.mean(), st.mean())
        ev = r["eval"]
        if "endings" in ev:
            e = ev["endings"]
            evs = (e["clear"], e["caught"], e["truncated"])
        else:   # 残機を記録する前の結果: 捕獲の報酬から捕まった割合を求める
            caught = -ev["reward_items_mean"].get("death", 0.0) / r["rewards"]["death"] if r["rewards"]["death"] else 0.0
            evs = (ev["clear_rate"], caught, max(0.0, 1 - ev["clear_rate"] - caught))
        groups.setdefault((r["exp"], r["label"], r.get("lives", 1)), []).append(tr + evs + (ev["steps_mean"],))
    rows = []
    for (exp, label, lives), v in sorted(groups.items()):
        m = np.mean(np.array(v), axis=0)
        rows.append([EXP_TITLE.get(exp, exp).split(" ")[0], label, lives, f"{m[0]:.0%}", f"{m[1]:.0%}", f"{m[2]:.0%}",
                     f"{m[3]:.0f}", f"{m[4]:.0%}", f"{m[5]:.0%}", f"{m[6]:.0%}", f"{m[7]:.0f}"])
    header = ["実験", "条件", "残機", "学習: クリア", "学習: 捕獲で終了", "学習: 打ち切り", "学習: 平均ステップ",
              "評価: クリア", "評価: 捕獲で終了", "評価: 打ち切り", "評価: 平均ステップ"]
    write_table(os.path.join(out, "figures", "endings"), header, rows,
                "エピソードの終わり方 (学習 = 直近100エピソード / 評価 = 学習後の貪欲方策。打ち切り = 1,000ステップの上限)", plt)


def report(args):
    plt = _plt()
    out = args.out
    results = []
    runs = os.path.join(out, "runs")
    if os.path.isdir(runs):
        for r in os.listdir(runs):
            p = os.path.join(runs, r, "result.json")
            if os.path.exists(p):
                with open(p, encoding="utf-8") as f:
                    results.append(json.load(f))
    report_design(plt, out)
    for exp in ("exp0", "exp1", "exp2", "exp3", "exp5", "exp6", "exp7"):
        report_experiment(plt, out, exp, results)
    report_pso(plt, out)
    report_rewards(plt, out, results)
    report_endings(plt, out, results)
    write_readme(out, results)


def write_readme(out, results):
    lines = ["# 報酬設計の実験 (自動生成)", "",
             "評価指標: **学習後の貪欲方策 (ランダムな行動なし) で、盤面の乱数を固定して複数回プレイした平均スコア**。"
             "補助指標: クリア率、学習中の直近100エピソード平均、平均ステップ数、Q値の振れ幅、損失。", ""]
    cp = os.path.join(out, "chain.json")
    if os.path.exists(cp):
        with open(cp, encoding="utf-8") as f:
            ch = json.load(f)
        lines += ["## 次の実験の基準値の決め方", ""]
        for c in ch["chain"]:
            lines.append(f"- {c['from']}: {c['chosen']} を採用 ({c['reason']})")
        lines.append("")
    lines += ["## ファイル", "",
              "- `figures/design/` 状態の構成・γ^T・ハイパーパラメータ一覧・数え上げ (facts.json)",
              "- `figures/rewards/` 報酬表 (前回 / トータル設計 / 単変量の最良 / PSO の最良)、報酬の内訳",
              "- `figures/exp0`〜`exp3`, `exp5` 条件ごとのグラフ (cond*.png)、重ね合わせ (overlay.png)、Q値の振れ幅 (q_range.png)、結果表 (results.*)",
              "- `figures/exp4_pso/` PSO の収束・各変数の位置・設定表・変数と結果",
              "- `runs/` 学習1本ごとのログ (log.csv) と結果 (result.json)、モデル (model.pt)", "",
              f"完了した学習: {len(results)} 本 ({time.strftime('%Y-%m-%d %H:%M')} 時点)"]
    with open(os.path.join(out, "README.md"), "w", encoding="utf-8") as f:
        f.write("\n".join(lines) + "\n")


def main():
    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    p.add_argument("--out", required=True)
    p.add_argument("--workers", type=int, default=max(1, (os.cpu_count() or 2) - 0))
    p.add_argument("--episodes", type=int, default=1000, help="単変量比較・最終比較の1本あたりの学習エピソード数")
    p.add_argument("--eval-episodes", type=int, default=20)
    p.add_argument("--seeds", type=int, nargs="+", default=[101, 202, 303])
    p.add_argument("--report-only", action="store_true")
    p.add_argument("--quick", action="store_true", help="動作確認 (数エピソード)")
    a = p.parse_args()
    if a.quick:
        a.episodes, a.eval_episodes, a.seeds = 4, 2, [101, 202]
    if a.report_only:
        report(a)
    else:
        main_pipeline(a)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile record_plays.py
"""
報酬設計の実験で学習したモデルのプレイ動画を作る (発表用)。
  python record_plays.py --out /content/drive/MyDrive/DQN_reward_study            # 実験5 の3条件
  python record_plays.py --out <dir> --exps exp5 exp6 exp7
出力 (<out>/videos/):
  compare_<exp>_board<k>.mp4  同じ盤面・同じ学習 seed で各条件を横に並べた動画 (先に終わった条件は最後の画面で止める)
  best_<exp>_<条件>.mp4       条件ごとに、3つの学習 seed × 評価の盤面20個 の中で一番点数の高かったプレイ
  plays_<exp>.csv             録画に使った各プレイの得点・終わり方・ステップ数 (実験ごとに別ファイル)
評価と同じく貪欲方策 (ランダムな行動なし)。盤面の乱数は reward_study.EVAL_SEED0 + k。
"""
import argparse
import csv
import glob
import json
import os

os.environ.setdefault("SDL_VIDEODRIVER", "dummy")   # 画面のない環境 (Colab) で pygame を使う

import numpy as np
from PIL import Image, ImageDraw, ImageFont

from pacman_env import PacManEnv
from reward_study import EVAL_SEED0, FIXED


def _font(size):
    for p in ("/usr/share/fonts/opentype/noto/NotoSansCJK-Bold.ttc", "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc",
              "C:/Windows/Fonts/meiryob.ttc", "C:/Windows/Fonts/msgothic.ttc"):
        if os.path.exists(p):
            return ImageFont.truetype(p, size)
    return ImageFont.load_default()


FONT, FONT_S = _font(22), _font(16)


def load(run_dir):
    from dqn_agent import load_agent
    with open(os.path.join(run_dir, "result.json"), encoding="utf-8") as f:
        r = json.load(f)
    env = PacManEnv(render_mode="rgb_array", ghost_mode=FIXED["ghost_mode"], num_ghosts=FIXED["num_ghosts"],
                    rewards=r["rewards"], lives=r.get("lives", 1))
    agent = load_agent(os.path.join(run_dir, "model.pt"), env.observation_space.shape[0], 4)
    return r, env, agent


def play(env, agent, board, frames=True):
    """1回プレイして (画像のリスト, 得点, 終わり方, ステップ数) を返す"""
    obs, _ = env.reset(seed=EVAL_SEED0 + board)
    imgs = []
    while True:
        q = agent.get_qvalues(obs)
        env.set_qvalues(q)                      # 盤面に行動ごとのQ値の矢印を描く
        if frames:
            imgs.append(env.render())
        obs, _, te, tr, info = env.step(int(np.argmax(q)))
        if te or tr:
            if frames:
                imgs.append(env.render())
            end = "クリア" if info["cleared"] else ("上限で打ち切り" if tr and not te else "捕獲")
            return imgs, info["score"], end, env.steps


def caption(img, title, sub):
    """上に条件名と結果の帯を付ける"""
    im = Image.fromarray(img)
    w, h = im.size
    out = Image.new("RGB", (w, h + 62), (20, 20, 40))
    out.paste(im, (0, 62))
    d = ImageDraw.Draw(out)
    d.text((8, 4), title, font=FONT, fill=(255, 230, 120))
    d.text((8, 36), sub, font=FONT_S, fill=(220, 220, 220))
    return np.array(out)


def save_mp4(frames, path, fps):
    import imageio.v2 as imageio
    imageio.mimsave(path, frames, fps=fps, macro_block_size=1)
    print("保存:", path, f"({len(frames)} フレーム)")


def runs_of(out, exp):
    rs = []
    for p in sorted(glob.glob(os.path.join(out, "runs", f"{exp}_*", "result.json"))):
        if os.path.exists(os.path.join(os.path.dirname(p), "model.pt")):
            with open(p, encoding="utf-8") as f:
                rs.append((json.load(f), os.path.dirname(p)))
    return rs


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--out", required=True, help="DQN_reward_study フォルダ")
    ap.add_argument("--exps", nargs="+", default=["exp5"])
    ap.add_argument("--boards", type=int, default=20, help="一番良いプレイを探す評価の盤面の数")
    ap.add_argument("--compare-seed", type=int, default=None,
                    help="並べ比べに使う学習 seed (省略時: クリアした条件の一番良いプレイの seed と盤面を使う)")
    ap.add_argument("--fps", type=int, default=8)
    a = ap.parse_args()
    vd = os.path.join(a.out, "videos")
    os.makedirs(vd, exist_ok=True)
    log = []
    for exp in a.exps:
        rs = runs_of(a.out, exp)
        if not rs:
            print("モデルがありません:", exp)
            continue
        labels = list(dict.fromkeys(r["label"] for r, _ in rs))
        # 条件ごとに一番良いプレイを探す (画像なしで速く回し、見つけた回だけ録画し直す)
        best_board = {}
        for lab in labels:
            best = None
            for r, d in rs:
                if r["label"] != lab:
                    continue
                _, env, agent = load(d)
                for k in range(a.boards):
                    _, sc, end, st = play(env, agent, k, frames=False)
                    log.append([exp, lab, r["seed"], k, sc, end, st])
                    key = (end == "クリア", sc)
                    if best is None or key > best[0]:
                        best = (key, d, r, k, end, st)
            (_, d, r, k, end, st) = best
            _, env, agent = load(d)
            imgs, sc, end, st = play(env, agent, k)
            sub = f"学習 seed {r['seed']} / 盤面 {k} / 得点 {sc} / {end} ({st} ステップ)"
            save_mp4([caption(f, f"{exp} {lab}（一番良いプレイ）", sub) for f in imgs],
                     os.path.join(vd, f"best_{exp}_{lab.replace(' ', '').replace('/', '_')}.mp4"), a.fps)
            best_board[lab] = (k, r["seed"], end)
        # 同じ盤面・同じ seed で横に並べる。クリアした条件があればその盤面と seed を使う (クリアの場面を見せるため)
        ref = next((lab for lab in labels if best_board[lab][2] == "クリア"), labels[-1])
        k, seed, _ = best_board[ref]
        seed = a.compare_seed or seed
        panels = []
        for lab in labels:
            hit = [(r, d) for r, d in rs if r["label"] == lab and r["seed"] == seed]
            if not hit:
                continue
            r, d = hit[0]
            _, env, agent = load(d)
            imgs, sc, end, st = play(env, agent, k)
            panels.append([caption(f, lab, f"得点 {sc} / {end} ({st} ステップ)") for f in imgs])
        n = max(len(p) for p in panels)
        frames = []
        for t in range(n + 2 * a.fps):          # 最後に2秒止める
            row = [p[min(t, len(p) - 1)] for p in panels]
            frames.append(np.concatenate([np.pad(x, ((0, 0), (0, 8), (0, 0))) for x in row], axis=1))
        save_mp4(frames, os.path.join(vd, f"compare_{exp}_seed{seed}_board{k}.mp4"), a.fps)
    with open(os.path.join(vd, f"plays_{'_'.join(a.exps)}.csv"), "w", newline="", encoding="utf-8-sig") as f:
        csv.writer(f).writerows([["実験", "条件", "学習seed", "盤面", "得点", "終わり方", "ステップ"]] + log)


if __name__ == "__main__":
    main()

## 動作確認（1分ほど）
数エピソードだけで全実験を通し、図表まで作れるか確かめる。結果は `/content/quick` に作る（Drive には保存しない）。

In [ ]:
!rm -rf /content/quick && python -u reward_study.py --out /content/quick --quick 2>&1 | grep -v -E 'Using device|Model saved' | tail -5

## 本番（実験0〜5）
実験0 報酬セット → 実験1 捕獲ペナルティ → 実験2 クリアボーナス → 実験3 γ → 実験4 PSO → 実験5 最終比較 → 実験6 残機 (1 vs 3) → 実験7 残機3でのクリアボーナス。終わった学習は飛ばすので、実験5 まで済んでいれば実験6・7 だけを実行する。
前の実験で最も評価スコアが高かった値を、次の実験の基準にする（理由は `README.md` と `chain.json` に残る）。

In [ ]:
WORKERS = os.cpu_count()
EPISODES = 1000          # 1本あたりの学習エピソード数 (単変量比較・最終比較)
!python -u reward_study.py --out {OUT} --workers {WORKERS} --episodes {EPISODES} 2>&1 | grep --line-buffered -v -E 'Using device|Model saved'

## 図表の確認（途中でも実行してよい）

In [ ]:
!python reward_study.py --out {OUT} --report-only > /dev/null 2>&1
import glob
from IPython.display import Image, Markdown, display
display(Markdown(open(f'{OUT}/README.md', encoding='utf-8').read()))
for p in sorted(glob.glob(f'{OUT}/figures/*/overlay.png')) + sorted(glob.glob(f'{OUT}/figures/*/results.png')) \
         + glob.glob(f'{OUT}/figures/exp4_pso/convergence.png') + glob.glob(f'{OUT}/figures/rewards/reward_table.png'):
    print(p.replace(OUT + '/', ''))
    display(Image(p, width=900))

## プレイ動画（発表用）
学習済みモデルで、条件ごとの一番良いプレイと、同じ盤面・同じ seed で各条件を横に並べた動画を
`DQN_reward_study/videos/` に作る（数分）。クリアした条件があれば、その盤面で並べる。

In [ ]:
!pip install -q pygame imageio-ffmpeg
!python record_plays.py --out {OUT} --exps exp5 exp6 exp7 2>&1 | grep -E 'mp4|ありません'